[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shulruferor-sudo/Advance-Data-Science/blob/main/projects/Bureau%20of%20Statistics/Assignment_Solution.ipynb)


# Occupational Classification: Survey Data and Model Evaluation

This notebook examines survey data and metadata, tests field-specific preparation decisions, and evaluates occupational classification models. Each transformation and modeling decision is tied to observable evidence and assessed against its alternatives.


In [1]:
# Load the tools needed to trace survey answers from raw data to evaluated occupation predictions.
import io
import hashlib
import urllib.request
import re
import json
import warnings
from pathlib import Path
import importlib.util
import subprocess
import sys
from urllib.request import urlopen


warnings.filterwarnings('ignore', category=UserWarning)

import numpy as np
import pandas as pd

pd.set_option('display.html.use_mathjax', False)
pd.set_option('display.max_columns', None)

def custom_style(df):
    """Format DataFrames for notebook display; source values remain unchanged."""
    return (df.style.set_table_styles([
        {'selector': 'th', 'props': [
            ('background-color', '#1e293b'), ('color', '#ffffff'),
            ('font-family', 'sans-serif'), ('font-size', '13px'),
            ('padding', '10px 15px'), ('text-align', 'left')
        ]},
        {'selector': 'td', 'props': [
            ('font-family', 'sans-serif'), ('font-size', '12px'),
            ('padding', '8px 15px'), ('border-bottom', '1px solid #e2e8f0')
        ]},
        {'selector': 'tr:hover', 'props': [('background-color', '#f1f5f9')]}
    ]).set_properties(**{'background-color': '#f8fafc'}))

pd.DataFrame._repr_html_ = lambda self: custom_style(self)._repr_html_()

import matplotlib.pyplot as plt
from IPython.display import display, HTML

from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold, GroupKFold, cross_val_predict
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import ComplementNB
from sklearn.dummy import DummyClassifier

from sklearn.metrics import accuracy_score, f1_score, classification_report

SEED = 42


### Load the LaBSE model

[LaBSE](https://huggingface.co/sentence-transformers/LaBSE) encodes text from different languages in a shared 768-dimensional space. L2-normalized embeddings have unit length, so their dot product equals cosine similarity.

The method compares each full value containing Latin letters with the distinct Hebrew-only values in the same field. It automatically replaces the full value with the highest-scoring Hebrew value when cosine similarity exceeds `alpha`, whose default is `0.5`. Otherwise, it preserves the original value. The returned column keeps its original index. This standardizes labels by semantic similarity, which alone does not establish that two descriptions identify the same workplace or role.


In [2]:
# Unify equivalent Hebrew and English workplace descriptions before TF-IDF splits them into unrelated features; inspect matches because semantic similarity can change meaning.
if importlib.util.find_spec("sentence_transformers") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "sentence-transformers==5.1.2", ])
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/LaBSE", device="cpu")
model.eval()
print("LaBSE loaded.")

def transform_column_with_labse(column, alpha=0.5):
    values = column.astype("string").str.strip().replace("", pd.NA)
    has_hebrew = values.str.contains(r"[א-ת]", na=False)
    has_latin = values.str.contains(r"[A-Za-z]", na=False)

    latin_cells = values.loc[has_latin]
    hebrew_names = sorted(values.loc[has_hebrew & ~has_latin].unique())

    vectors = model.encode(latin_cells.tolist() + hebrew_names, batch_size=8, normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=False)

    similarities = vectors[:len(latin_cells)] @ vectors[len(latin_cells):].T
    best_indices = similarities.argmax(axis=1)
    best_scores = similarities[np.arange(len(latin_cells)), best_indices]
    best_hebrew = pd.Series(np.asarray(hebrew_names)[best_indices], index=latin_cells.index, dtype="string")

    transformed_column = column.copy()
    transformed_column.loc[latin_cells.index] = best_hebrew.where(best_scores > alpha, column.loc[latin_cells.index])
    return transformed_column

LaBSE loaded.


In [3]:
# Apply the same verified spelling and terminology corrections consistently across records.
text_transformations = {"קרית גת": "קריית גת", "לא יכול לפרט": "חסוי"}

def transform_text(value):
    return text_transformations.get(value, value)

## Source data

Load the survey responses and their metadata from the two workbook sheets without altering the source values.


In [4]:
# Which questionnaire fields describe the occupation, its context, and its recorded code? Read the metadata before interpreting values.
URL = "https://raw.githubusercontent.com/shulruferor-sudo/Advance-Data-Science/main/Datasets/cbs_occupation_survey_sample/DUMMY_DATASET.xlsx"
SHA = "e1e7f391c0675dd14830eaae6b5cd114edc256f18b956bb48a454b0b1209e1ac"
raw = urllib.request.urlopen(URL, timeout=90).read()

data = (pd.read_excel(io.BytesIO(raw), sheet_name="Survey Dataset").dropna(axis=0, how="all").dropna(axis=1, how="all"))

metadata = (pd.read_excel(io.BytesIO(raw), sheet_name="Metadata").dropna(axis=0, how="all").dropna(axis=1, how="any"))

display(metadata)
print()

display(data.head(20))
print()

,משתנה,הסבר,פורמט
0,ID,מספר רץ,INTEGER
1,YeshuvAvoda,שם יישוב עבודה,TEXT
2,ShemAvoda,שם מקום העבודה שלך,TEXT
3,SugAvoda,הפעילות העיקרית של מקום העבודה,TEXT
4,ShemMachlaka,אגף או מחלקה שהנסקר עובד,TEXT
5,SugMachlaka,הפעילות העיקרית של האגף או המחלקה שהנסקר עובד,TEXT
6,EzoAvoda,עבודה שהנסקר עושה במקום עבודה זה,TEXT
7,TeurPeula,הפעולות העיקריות שהנסקר עושה במקום עבודה,TEXT
8,TeurTafkid,מהו תואר המשרה של הנסקר במקום עבודה זה (במידה ויש),TEXT
9,MenahelEtMi,האם הנסקר מנהל או ממונה על עובדים אחרים?,INTEGER


,ID,YeshuvAvoda,ShemAvoda,SugAvoda,ShemMachlaka,SugMachlaka,EzoAvoda,TeurPeula,TeurTafkid,MenahelEtMi,MaamadAvoda,MakorSachar,TeudaGvoha,shnotlimud,Gil,TarichHathala,SemelAnafSofi,SemelMishlachSofi
0,1.000000,ירושלים,"למ""ס","איסוף, עיבוד ופרסום נתונים",מיקרו כלכלה,טיפול בנתוני מיקרו,סגן מנהל אגף,"כנ""ל",סגן מנהל אגף,3.000000,1.000000,1.000000,6.000000,22.000000,55.000000,2020-06-15 00:00:00,8411,2120
1,2.000000,ירושלים,בית חולים הדסה עין כרם,טיפול בחולים,רדיואתרפיה,טיפול בחולי סרטן,טכנאית רדיותרפיה,"כנ""ל","כנ""ל",nan,1.000000,1.000000,6.000000,22.000000,55.000000,2022-07-01 00:00:00,8610,3311
2,3.000000,באר שבע,קריית הממשלה,מתן שירות למשתמשים באתר GOV,תמיכה בלקוחות,"כנ""ל",נותנת תמיכה ללקוחות שמבקשים עזרה,מסבירה איך להשתמש בשירותים ממשלתיים באתר האינטרנט,nan,nan,1.000000,1.000000,5.000000,15.000000,29.000000,2021-06-06 00:00:00,8411,3512
3,4.000000,תל אביב,משרד החינוך,בית ספר יסודי,שמירה,nan,שומר בבית הספר,nan,nan,nan,1.000000,2.000000,3.000000,12.000000,48.000000,2024-07-01 00:00:00,8010,5414
4,5.000000,תל אביב,אמיר כהן,מטפל בקשיש במשפחה,nan,nan,מטפל בקשיש,"נותן אוכל, עוזר בניקיון בית, עוזר להתרחץ",nan,nan,1.000000,1.000000,4.000000,14.000000,42.000000,2010-01-11 00:00:00,XXXX,5322
5,6.000000,חיפה,משטרה,שוטר,nan,nan,שוטר,nan,nan,nan,1.000000,1.000000,4.000000,14.000000,35.000000,2022-05-31 00:00:00,8423,5412
6,7.000000,קרית גת,אינטל,ייצור צ'יפים למחשבים,חדר נקי,nan,עובד בחדר נקי,מפקח על תהליך ייצור צ'יפים,nan,nan,1.000000,1.000000,5.000000,15.000000,65.000000,1999-07-07 00:00:00,2610,3122
7,8.000000,חיפה,אינטל,פיתוח תוכנה לצ'יפים,הנדסת תוכנה,nan,"מהנדס תוכנה, מנהל צוות עובדים",מפתח קוד לצ'יפים,nan,1.000000,1.000000,1.000000,6.000000,18.000000,55.000000,2000-11-10 00:00:00,7201,2512
8,9.000000,אשקלון,שופר סל,קופאית,nan,nan,קופאית,nan,nan,nan,1.000000,3.000000,2.000000,11.000000,52.000000,2022-11-11 00:00:00,4711,5230
9,10.000000,הרצליה,EMILIO,"הייטק, פיתוח תוכנה",nan,nan,מתכנת,nan,nan,nan,1.000000,1.000000,5.000000,16.000000,24.000000,2021-12-14 00:00:00,6201,2514


### Field-by-field investigation


### Analysis of ID column

The metadata defines the values within the column ID as a values of a running index.
The metadata defines column ID to be an Integer type.

What exactly does the running index identify, the respondent, the questionnaire, a submission, or a record?

The assignment mantions explicitly an framework that allaws resubmissions from the same participant within the the same servey timeline, which means the system must have a key per a single person within a single survey, and the only one can provide that in the data is an ID column.

In [5]:
# What does each row and column represent? Confirm the record structure before investigating individual features.
data["ID"].notna().sum()

Out[5]: 200


Conclussion: The data analysis confirms the 200 records of surveys

### Analysis of YeshuvAvoda column

Was the job location location was obtained throgh predifined categories or filled in as a free text?
The metadata states that it is stored in a text type field, but there is no mention of haw it was obtained.
From abserving the data it is clear that most likely come from dropdawn list or chousen category becouse the only iregulaarities in the column are bouth valid verssions of the same city, just one older verssion and the other is from newear national regestery verssion. 
Most likely bouth verssions still remained, so I would raise this issue with the relavent stakeholders and peers.

In [6]:
# Which answers are absent, and is absence meaningful for that field? Measure missingness before deciding how to encode it.
data.loc[data["YeshuvAvoda"].isna() | data["YeshuvAvoda"].astype("string").str.strip().eq("").fillna(False), ["ID", "YeshuvAvoda"]]

Out[6]: Series([], Name: YeshuvAvoda, dtype: object)


Conclussion: The data analysis confirms that the fieald is a mandotory field which is also aligngs perfectly with my initial asamption.

### Analysis of ShemAvoda column

The metadata defines ShemAvoda as the name of the participant’s workplace.

How many records are empty, how many records are distinct relative to records that not empty, haw match records contain a Latin letters, and haw many records contain a mix of Hebrew and English letters in one cell?


In [7]:
shem_avoda = data.loc[data["ID"].notna(), "ShemAvoda"].astype("string").str.strip().replace("", pd.NA)

print("Empty records:", shem_avoda.isna().sum())
print("Distinct values among nonempty records:", shem_avoda.nunique(), "of", shem_avoda.count())

has_hebrew = shem_avoda.str.contains(r"[א-ת]", na=False)
has_latin = shem_avoda.str.contains(r"[A-Za-z]", na=False)
print("Records containing Hebrew letters:", has_hebrew.sum())
print("Records containing Latin letters:", has_latin.sum())
print("Records containing both Hebrew and Latin letters:", shem_avoda[has_hebrew & has_latin])

Empty records: 0
Distinct values among nonempty records: 82 of 200
Records containing Hebrew letters: 187
Records containing Latin letters: 14
Records containing both Hebrew and Latin letters: 139    יועצת IT עצמאית
Name: ShemAvoda, dtype: string


Conclusion: None of the 200 workplace records is empty. There are 82 distinct values among the 200 nonempty records (41%). Fourteen records contain Latin letters, and one of those also contains Hebrew letters in the same cell.

In [8]:
# Do bilingual employer names fragment the same workplace into different TF-IDF terms? Inspect normalization changes before encoding.
transformed_data = data.copy()
transformed_data["ShemAvoda"] = transform_column_with_labse(data["ShemAvoda"])

review = data.loc[shem_avoda.loc[has_latin].index, ["ID", "ShemAvoda"]].copy()
review["Transformed"] = transformed_data.loc[review.index, "ShemAvoda"]
display(review)

,ID,ShemAvoda,Transformed
9,10.000000,EMILIO,EMILIO
14,15.000000,NVIDIA,נבידיה
37,38.000000,INTEL,אינטל
46,47.000000,NVIDIA,נבידיה
55,56.000000,COGNYTE,COGNYTE
72,73.000000,EMILIO,EMILIO
77,78.000000,NVIDIA,נבידיה
100,101.000000,INTEL,אינטל
103,104.000000,EMILIO,EMILIO
109,110.000000,NVIDIA,נבידיה


The LaBSE method replaces a full value containing Latin letters with the highest-scoring existing Hebrew-only value in the same field when cosine similarity exceeds `alpha`. Otherwise, it retains the original value.

### Numerical representation of YeshuvAvoda

Complete the locality decision established above by merging `קרית גת` with `קריית גת` and applying the one-hot encoding specified in the metadata interpretation. All 200 locality values are populated, so this field requires no null replacement. Fit the category mapping on complete-label records and retain IDs solely for alignment.

In [9]:
# Do workplace-location spellings split equivalent locations? Standardize their representation before feature construction.
survey = transformed_data.loc[transformed_data["ID"].notna()].copy()
feature_fit_rows = survey["SemelMishlachSofi"].astype("string").str.replace(r"\.0$", "", regex=True).str.fullmatch(r"[0-9]{4}", na=False)

locality_values = survey["YeshuvAvoda"].astype("string").str.strip().map(transform_text, na_action="ignore").to_frame("YeshuvAvoda")

from sklearn.preprocessing import OneHotEncoder
locality_encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=True)
locality_encoder.fit(locality_values.loc[feature_fit_rows])
locality_matrix = locality_encoder.transform(locality_values)
locality_features = pd.DataFrame.sparse.from_spmatrix(locality_matrix, index=survey.index, columns=locality_encoder.get_feature_names_out(["YeshuvAvoda"]))
print("YeshuvAvoda one-hot matrix:", locality_matrix.shape)

YeshuvAvoda one-hot matrix: (200, 20)


### TF-IDF representation of ShemAvoda

For `ShemAvoda`, retain TF-IDF without L2 normalization because workplace names can share words while differing through additional identifying terms. With ordinary term counts, an added term contributes another weighted feature without rescaling the shared words. This preserves weighted text magnitude as potential classification information. Whether that information improves prediction must be assessed against L2-normalized TF-IDF within training-fold validation.

Lowercase the text after LaBSE, retain Hebrew and Latin words and single-letter initials, and use unsmoothed IDF. The matrix below describes the current survey. Vocabulary and IDF must be fitted within each training partition when evaluating a classifier. [scikit-learn TF-IDF documentation](https://scikit-learn.org/stable/modules/feature_extraction.html#tfidf-term-weighting)


In [10]:
# Employer names may appear in Hebrew or English; standardize equivalent names before TF-IDF assigns separate terms.
shem_avoda_transformed = transformed_data.loc[transformed_data["ID"].notna(), "ShemAvoda"].astype("string").str.strip().map(transform_text, na_action="ignore")

shem_avoda_vectorizer = TfidfVectorizer(lowercase=True, token_pattern=r"(?u)\b\w+\b", binary=False, sublinear_tf=False, smooth_idf=False, norm=None)
shem_avoda_vectorizer.fit(shem_avoda_transformed.loc[feature_fit_rows])
shem_avoda_tfidf = shem_avoda_vectorizer.transform(shem_avoda_transformed)

shem_avoda_terms = shem_avoda_vectorizer.get_feature_names_out()
print("ShemAvoda TF-IDF matrix:", shem_avoda_tfidf.shape)
print("Nonzero weights:", shem_avoda_tfidf.nnz)

ShemAvoda TF-IDF matrix: (200, 108)
Nonzero weights: 365


**Conclusion:** The workplace-name column is represented numerically by TF-IDF weights after the LaBSE transformation. All 200 workplace names are populated, so this field needs no null replacement. Its vocabulary and IDF are fitted on complete-label records. Classifier validation must refit these learned quantities inside each training fold.

## From workplace normalization to occupational evidence

The assignment asks for an occupation prediction from a questionnaire that can be submitted with unanswered fields. The remaining columns describe different things: the organization, the participant's own work, employment arrangements, education, age, dates, and existing classifications. Those differences determine what can be inferred from a response and what preparation it needs.

The investigation below starts from the workplace transformation above. Each field is interpreted against its metadata and the assignment, examined using the recorded answers, and then prepared according to the resulting conclusion. The original answers remain available alongside the prepared values.

In [11]:
# How can employer and location context be combined without obscuring their separate meanings? Align their feature blocks by record.
preparation_audit = survey[["ID"]].copy()

shem_avoda_features = pd.DataFrame.sparse.from_spmatrix(shem_avoda_tfidf, index=survey.index, columns="ShemAvoda__" + shem_avoda_vectorizer.get_feature_names_out())
prepared_data = survey.drop(columns=["ShemAvoda", "YeshuvAvoda"]).join(shem_avoda_features).join(locality_features)
field_encoders = {"YeshuvAvoda": locality_encoder, "ShemAvoda": shem_avoda_vectorizer}

### Analysis of SugAvoda column

The metadata defines `SugAvoda` as the workplace's main activity. This is an open description of what the organization does. Whole-answer categories would treat related phrases as unrelated labels, while TF-IDF can retain shared activity terms and the words that distinguish them.

Which recorded values describe an activity, and which provide no usable description? Inspect the answer frequencies and full values containing Latin letters before defining the treatment.

In [12]:
# Does employer activity describe the organization rather than the respondent? Inspect its wording before using it as contextual evidence.
workplace_activity = survey["SugAvoda"].astype("string").str.strip().replace("", pd.NA)
display(workplace_activity.value_counts(dropna=False).rename("Records"))

display(workplace_activity.loc[workplace_activity.str.contains(r"[A-Za-z]", na=False)].to_frame())

SugAvoda
<NA>                 25
מסעדה                 7
בית ספר יסודי         7
חנות למכירת בגדים     7
אירוח                 6
                     ..
מכונאות               1
מלצר                  1
חבלן                  1
גננת                  1
סטטיסטיקה             1
Name: Records, Length: 79, dtype: Int64

,SugAvoda
2,מתן שירות למשתמשים באתר GOV
65,מתן שירות למשתמשים באתר GOV
139,יועצת IT


**Conclusion and preparation:** The 175 populated answers contain 78 distinct values, including workplace types, activities, and job descriptions. Twenty-five answers are blank. The assignment permits partially answered questionnaires, so a blank does not justify inventing an activity or deleting the record. Use empty text only for an unanswered cell.

Retain all recorded wording, including the four `חסוי` responses and the three Latin-containing values. `חסוי` records that the participant treats the workplace activity as confidential. This supplies context that an omitted answer does not, so its term remains available to the classifier. Encode this field with the same TF-IDF settings as `ShemAvoda`, using its own vocabulary. Fit vocabulary and IDF within each training fold during validation. [TF-IDF documentation](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html)

In [13]:
# How can employer activity help classification without replacing the respondent’s own work? Build a separate text representation.
activity_text = workplace_activity.map(transform_text, na_action="ignore").fillna("")

activity_vectorizer = clone(shem_avoda_vectorizer)
activity_vectorizer.fit(activity_text.loc[feature_fit_rows])

activity_matrix = activity_vectorizer.transform(activity_text)
activity_features = pd.DataFrame.sparse.from_spmatrix(activity_matrix, index=survey.index, columns="SugAvoda__" + activity_vectorizer.get_feature_names_out())

prepared_data = prepared_data.drop(columns="SugAvoda").join(activity_features)
field_encoders["SugAvoda"] = activity_vectorizer

print("SugAvoda TF-IDF matrix:", activity_matrix.shape)

SugAvoda TF-IDF matrix: (200, 116)


### Analysis of ShemMachlaka column

The metadata asks for the division or department in which the participant works. Department names may combine a broad function with a specialization, so individual words can carry useful information beyond the identity of the complete answer. A blank records an unanswered question and does not establish that no department exists.

Do the observed answers form a documented category set, or do they contain varied descriptions and abbreviations? Inspect their frequencies to decide how much wording the representation must retain.

In [14]:
# Does the department name add useful context, and how often is it answered? Examine coverage and distinct values first.
department_names = survey["ShemMachlaka"].astype("string").str.strip().replace("", pd.NA)
display(department_names.value_counts(dropna=False).rename("Records"))

ShemMachlaka
<NA>               64
שמירה               7
תמיכה בלקוחות       6
קוסמטיקה וטיפוח     6
משק                 6
                   ..
חשמלאות             1
יומנאי              1
מכונאות             1
מסעדה               1
חבלן                1
Name: Records, Length: 62, dtype: Int64

**Conclusion and preparation:** The 136 populated answers contain 61 distinct values, including department names, functions, job wording, and six Latin-containing entries. Four answers are `חסוי`, while 64 are blank. The response `חסוי` indicates that the participant treats the department information as confidential. This provides context about the department, even though it does not disclose its name, and differs from an unanswered question. Retain it as a text feature. A blank does not establish that no department exists. Use empty text only for missing answers and encode every recorded value with this field’s TF-IDF vocabulary.

In [15]:
# Can department names contribute context independently of job descriptions? Encode them in a separate feature block.
department_text = department_names.map(transform_text, na_action="ignore").fillna("")

department_vectorizer = clone(shem_avoda_vectorizer)
department_vectorizer.fit(department_text.loc[feature_fit_rows])

department_matrix = department_vectorizer.transform(department_text)
department_features = pd.DataFrame.sparse.from_spmatrix(department_matrix, index=survey.index, columns="ShemMachlaka__" + department_vectorizer.get_feature_names_out())
prepared_data = prepared_data.drop(columns="ShemMachlaka").join(department_features)
field_encoders["ShemMachlaka"] = department_vectorizer

print("ShemMachlaka TF-IDF matrix:", department_matrix.shape)

ShemMachlaka TF-IDF matrix: (200, 77)


### Analysis of SugMachlaka column

The metadata defines `SugMachlaka` as the department’s main activity. Each row is one participant’s questionnaire. The participant may write `כנ"ל`, meaning “the same,” after the preceding answer instead of repeating its text. Here the preceding field is `ShemMachlaka`, so that row’s department name supplies the repeated answer.

Inspect the recorded activities and references before completing the text transformation.

In [16]:
# Are department-activity responses actual descriptions or references to other answers? Inspect repeated placeholders before vectorization.
department_activity = survey["SugMachlaka"].astype("string").str.strip().replace("", pd.NA)
display(department_activity.value_counts(dropna=False).rename("Records"))

SugMachlaka
<NA>                           134
כנ"ל                             8
טיפול בחולי סרטן                 4
טיפול באנשים עם בעיות            4
משחק                             4
אריזה                            3
טיפול בלקוחות עסקיים             3
תמיכה במערכות מחשוב של הבנק      3
חישובי שכר לעובדים               3
טיפול בנתוני מיקרו               2
הזמנת ציוד                       2
R&D                              2
רוקח                             2
אשראי                            2
סיעוד                            2
קבלת קהל                         2
טיפול בנתוני מאקרו               1
חשבת שכר לעובדים                 1
איסוף נתונים בשטח                1
טיפול בבקשת הלואות               1
שיווק עוגות                      1
מחקר ופיתוח                      1
מחסן                             1
משאבי אנוש                       1
IT                               1
מדע נתונים                       1
חפירות                           1
שיווק                            1
רואה חשב

**Conclusion and preparation:** The field has 134 blanks and 66 populated answers, including eight `כנ"ל` references. Copy `ShemMachlaka` from the same row into those eight references because the participant has explicitly repeated the preceding answer. Keep genuinely unanswered cells empty. Then apply the existing LaBSE method to this column with its default `alpha=0.5` before TF-IDF. The method selects Hebrew values already present in this feature.

In [17]:
# Would a reference such as “same as above” become a meaningless token? Resolve supported within-record references before encoding.
department_activity_text = department_activity.copy()
department_activity_text.loc[department_activity_text.eq('כנ"ל')] = department_names
department_activity_text = department_activity_text.map(transform_text, na_action="ignore").fillna("")

department_activity_text = transform_column_with_labse(department_activity_text)

department_activity_vectorizer = clone(shem_avoda_vectorizer)
department_activity_vectorizer.fit(department_activity_text.loc[feature_fit_rows])

department_activity_matrix = department_activity_vectorizer.transform(department_activity_text)
department_activity_features = pd.DataFrame.sparse.from_spmatrix(department_activity_matrix, index=survey.index, columns="SugMachlaka__" + department_activity_vectorizer.get_feature_names_out())
prepared_data = prepared_data.drop(columns="SugMachlaka").join(department_activity_features)
field_encoders["SugMachlaka"] = department_activity_vectorizer
print("SugMachlaka TF-IDF matrix:", department_activity_matrix.shape)

SugMachlaka TF-IDF matrix: (200, 58)


### Analysis of EzoAvoda column

The metadata asks what work the participant performs. The answer can state an occupation or describe duties. A non-disclosure response records a decision to withhold details, while a blank supplies no response. Preserve that distinction and the participant’s wording.

Inspect the descriptions, non-disclosure responses, and Latin-containing values to determine the preparation required.

In [18]:
# Which words describe what the respondent actually does? Inspect this primary occupation signal before feature engineering.
work_descriptions = survey["EzoAvoda"].astype("string").str.strip().replace("", pd.NA)
display(work_descriptions.value_counts(dropna=False).rename("Records"))

EzoAvoda
שומר בבית הספר                7
<NA>                          7
טבח                           7
קוסמטיקאית                    6
מנקה בית ספר אחרי הלימודים    6
                             ..
בדיקת דירוג אשראי             1
מציל בבריכה                   1
מכונאי של רכבים חשמליים       1
חבלן                          1
סימול סקרים                   1
Name: Records, Length: 85, dtype: Int64

**Conclusion and preparation:** There are 193 populated answers, seven blanks, and 84 distinct populated values. Two answers say `חסוי` and two say `לא יכול לפרט`. Standardize `לא יכול לפרט` to `חסוי` because both express non-disclosure. The response records that the participant treats details of the work as confidential. Retaining it gives the classifier that context while keeping it distinct from an unanswered question. The seven blanks cannot be interpreted as unemployment because the assignment permits omitted answers. Convert only missing values to empty text. Apply the existing LaBSE method with its default `alpha=0.5`, then encode the returned values with TF-IDF.

In [19]:
# Occupation descriptions provide direct evidence of the work performed; represent them separately from employer activity.
work_description_text = work_descriptions.map(transform_text, na_action="ignore").fillna("")

work_description_text = transform_column_with_labse(work_description_text)

work_description_vectorizer = clone(shem_avoda_vectorizer)
work_description_vectorizer.fit(work_description_text.loc[feature_fit_rows])

work_description_matrix = work_description_vectorizer.transform(work_description_text)
work_description_features = pd.DataFrame.sparse.from_spmatrix(work_description_matrix, index=survey.index, columns="EzoAvoda__" + work_description_vectorizer.get_feature_names_out())
prepared_data = prepared_data.drop(columns="EzoAvoda").join(work_description_features)
field_encoders["EzoAvoda"] = work_description_vectorizer
print("EzoAvoda TF-IDF matrix:", work_description_matrix.shape)

EzoAvoda TF-IDF matrix: (200, 158)


### Analysis of TeurPeula column

The metadata asks for the main actions performed at work. Verbs and the objects of those actions distinguish duties, so retaining phrase content is more useful than assigning a separate category to every answer. Each row belongs to one participant. When the participant writes `כנ"ל`, meaning “the same,” the task answer repeats the preceding `EzoAvoda` answer in that row.

Inspect the reported tasks and reference expressions before completing the transformation.

In [20]:
# Are task descriptions substantive or shortcuts referring to other fields? Identify references before treating them as work evidence.
task_descriptions = survey["TeurPeula"].astype("string").str.strip().replace("", pd.NA)
display(task_descriptions.value_counts(dropna=False).rename("Records"))

TeurPeula
<NA>                                                 136
כנ"ל                                                  12
מפקח על תהליך ייצור צ'יפים                             4
מסבירה לאנשים לגבי מוצרים של חברת LOREAL               4
נותן תמיכה לעובדים בבנק ופוטר בעיות תקשורת             3
עוזר לחשמלאי                                           3
טכנאי מזגנים                                           3
קבלת קהל                                               3
נותן אוכל, עוזר בניקיון בית, עוזר להתרחץ               2
מפתח קוד לצ'יפים                                       2
מאכיל ומנקה                                            2
בחינת יכולת החזר                                       2
פיתוח קוד                                              2
טיפול יום יומי בצרכים של הקשיש                         2
מסבירה איך להשתמש בשירותים ממשלתיים באתר האינטרנט      2
בונה מודלים של AI                                      1
עוזר לתקן צינורות                                      1
חישובי שכר לעובדים   

**Conclusion and preparation:** Of 64 populated answers, 12 are `כנ"ל` and 52 explicitly describe tasks. Another 136 answers are blank. An unanswered task field does not establish that the participant performs no tasks, so leave genuine blanks empty. For each `כנ"ל`, copy the resolved `EzoAvoda` text from the same row because the participant has stated that the answer is the same. Apply the existing LaBSE method to the completed task column with its default `alpha=0.5`, then perform TF-IDF.

In [21]:
# How can task evidence be retained when respondents refer to another answer? Resolve supported same-record references before vectorization.
task_text = task_descriptions.copy()
task_text.loc[task_text.eq('כנ"ל')] = work_description_text
task_text = task_text.map(transform_text, na_action="ignore").fillna("")

task_text = transform_column_with_labse(task_text)

task_vectorizer = clone(shem_avoda_vectorizer)
task_vectorizer.fit(task_text.loc[feature_fit_rows])

task_matrix = task_vectorizer.transform(task_text)
task_features = pd.DataFrame.sparse.from_spmatrix(task_matrix, index=survey.index, columns="TeurPeula__" + task_vectorizer.get_feature_names_out())
prepared_data = prepared_data.drop(columns="TeurPeula").join(task_features)
field_encoders["TeurPeula"] = task_vectorizer
print("TeurPeula TF-IDF matrix:", task_matrix.shape)

TeurPeula TF-IDF matrix: (200, 105)


### Analysis of TeurTafkid column

The metadata asks for a job title if one exists. An unanswered optional title does not establish that the participant lacks an occupation. Titles can contain qualifiers such as deputy or manager, so retain their full wording.

Inspect the recorded titles and reference expressions before encoding this field.

In [22]:
# Does the job title add evidence beyond the description of actual work, and how often is it present? Inspect it separately.
job_titles = survey["TeurTafkid"].astype("string").str.strip().replace("", pd.NA)
display(job_titles.value_counts(dropna=False).rename("Records"))

TeurTafkid
<NA>                                  172
כלכלן                                   6
כנ"ל                                    4
סגן מנהל אגף                            3
טכנאי מחשבים                            3
0                                       1
בונה מודלים של AI                       1
סוקר                                    1
פיתוח תכנה                              1
מנהל פיתןח של 2 מוצרים                  1
מנהלת משאבי אנוש של עובדיםן בישראל      1
יועצת IT                                1
סמנכ"ל שיווק                            1
תמיכה במחשוב                            1
מנהלת משאבי אנוש                        1
יועץ השקעות                             1
פקיד                                    1
Name: Records, dtype: Int64

**Conclusion and preparation:** There are 172 blanks and 28 populated answers. A blank may reflect either no formal title or an omitted answer, so neither case justifies inventing a title. Four answers are `כנ"ל` and one is `0`. Copy the resolved `TeurPeula` answer into each reference, working from left to right within that participant’s row. Thus `טכנאית רדיותרפיה | כנ"ל | כנ"ל` becomes three occurrences of `טכנאית רדיותרפיה`. Retain `0` as recorded because its meaning has not been established. Convert only missing values to empty text and encode the completed title field with TF-IDF.

In [23]:
# Can a stated title help when present without inventing one when absent? Encode this optional field separately.
title_text = job_titles.copy()
title_text.loc[title_text.eq('כנ"ל')] = task_text

title_text = title_text.map(transform_text, na_action="ignore").fillna("")

title_vectorizer = clone(shem_avoda_vectorizer)
title_vectorizer.fit(title_text.loc[feature_fit_rows])

title_matrix = title_vectorizer.transform(title_text)
title_features = pd.DataFrame.sparse.from_spmatrix(title_matrix, index=survey.index, columns="TeurTafkid__" + title_vectorizer.get_feature_names_out())
prepared_data = prepared_data.drop(columns="TeurTafkid").join(title_features)
field_encoders["TeurTafkid"] = title_vectorizer
print("TeurTafkid TF-IDF matrix:", title_matrix.shape)

TeurTafkid TF-IDF matrix: (200, 34)


### Analysis of MenahelEtMi column

The metadata records supervisory responsibility using three codes: 1 for employees, 2 for managers, and 3 for both. Treat the responses as ordered management levels and interpret an unanswered question as no managerial responsibility. Encode this as zero and preserve levels 0–3 in one numeric feature.

Count the recorded levels and missing answers to establish how many records receive the zero encoding.

In [24]:
# What do the management codes mean and how are they distributed? Check the metadata and responses before using responsibility as context.
management_level = survey["MenahelEtMi"]
display(management_level.value_counts(dropna=False).sort_index().rename("Records"))

MenahelEtMi
1.0      8
2.0     11
3.0     13
NaN    168
Name: Records, dtype: int64

**Conclusion and preparation:** Codes 1, 2, and 3 occur in 8, 11, and 13 records respectively. Assign zero to the 168 unanswered records and retain the recorded levels for the other 32. Zero represents the interpretation of no managerial responsibility. Keep the original responses in the source table.

In [25]:
# How can management responsibility contribute without treating response codes as measured quantities? Encode the categories as categories.
prepared_data["MenahelEtMi"] = management_level.fillna(0).astype("int64")

### Analysis of MaamadAvoda column

The metadata defines seven employment arrangements: employee, self-employed with one or two employees, self-employed with at least three employees, self-employed without employees, cooperative member, kibbutz member, and unpaid family worker. The code numbers identify arrangements rather than measuring a quantity, so this field requires one-hot encoding.

Inspect the observed categories and unanswered records before preparing the categorical input.

In [26]:
# Which employment statuses determine whether follow-up questions apply? Inspect their distribution before interpreting salary-source answers.
employment_status = survey["MaamadAvoda"]
display(employment_status.value_counts(dropna=False).sort_index().rename("Records"))

MaamadAvoda
1.0    177
2.0      1
4.0     10
6.0     10
NaN      2
Name: Records, dtype: int64

**Conclusion and preparation:** The recorded employment-status codes identify different arrangements, not quantities. There are two unanswered records. Assign these the additional category `0`, meaning an unanswered employment-status question, and retain every recorded code. One-hot encode the categories separately so the added code does not impose an ordering or distance. Keep every category column (`drop=None`) and handle unseen categories without changing the fitted vocabulary. [scikit-learn OneHotEncoder](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html).

Use the codebook categories explicitly so an allowed response retains its own column even when absent from a training fold.

In [27]:
# Can employment arrangement add context without implying numerical order? Encode the status categories separately.
employment_categories = employment_status.fillna(0).astype("Int64").astype("string")

employment_encoder = OneHotEncoder(categories=[["0", "1", "2", "3", "4", "5", "6", "7"]], handle_unknown="ignore", sparse_output=True)
employment_encoder.fit(employment_categories.loc[feature_fit_rows].to_frame("MaamadAvoda"))

employment_matrix = employment_encoder.transform(employment_categories.to_frame("MaamadAvoda"))
employment_features = pd.DataFrame.sparse.from_spmatrix(employment_matrix, index=survey.index, columns=employment_encoder.get_feature_names_out(["MaamadAvoda"]))
prepared_data = prepared_data.drop(columns="MaamadAvoda").join(employment_features)
field_encoders["MaamadAvoda"] = employment_encoder
print("MaamadAvoda one-hot matrix:", employment_matrix.shape)

MaamadAvoda one-hot matrix: (200, 8)


### Analysis of MakorSachar column

The metadata defines four salary sources and restricts this question to employees and cooperative members, corresponding to `MaamadAvoda` codes 1 and 5. Salary source is nominal and requires one-hot encoding. A blank can mean that the question does not apply or that an applicable answer was omitted. Its treatment therefore depends on the questionnaire's explicit routing condition.

Count salary-source responses and apply that condition to distinguish unanswered, non-applicable, and contradictory records.

In [28]:
# Are salary-source blanks inapplicable, unanswered, or inconsistent with employment status? Check the routing logic before preparation.
salary_source = survey["MakorSachar"]
display(salary_source.value_counts(dropna=False).sort_index().rename("Records"))

salary_status_known = survey["MaamadAvoda"].notna()
salary_eligible = survey["MaamadAvoda"].isin([1, 5])
salary_ineligible = salary_status_known & ~salary_eligible
display(pd.Series({"Eligible with an answer": int((salary_eligible & salary_source.notna()).sum()), "Ineligible without an answer": int((salary_ineligible & salary_source.isna()).sum()), "Ineligible with an answer": int((salary_ineligible & salary_source.notna()).sum()), "Unknown eligibility without an answer": int((~salary_status_known & salary_source.isna()).sum())}))

salary_eligibility_issues = survey.loc[salary_ineligible & salary_source.notna(), ["ID", "MaamadAvoda", "MakorSachar"]]
display(salary_eligibility_issues)

MakorSachar
1.0    144
2.0     26
3.0     11
4.0      5
NaN     14
Name: Records, dtype: int64

Eligible with an answer                  177
Ineligible without an answer              12
Ineligible with an answer                  9
Unknown eligibility without an answer      2
dtype: int64

,ID,MaamadAvoda,MakorSachar
33,34.000000,4.000000,4.000000
48,49.000000,4.000000,1.000000
111,112.000000,4.000000,1.000000
139,140.000000,4.000000,1.000000
165,166.000000,4.000000,1.000000
175,176.000000,4.000000,1.000000
186,187.000000,4.000000,1.000000
189,190.000000,2.000000,1.000000
197,198.000000,4.000000,1.000000


**Conclusion and preparation:** The field contains 186 recorded answers and 14 blanks. Use the existing category `4 — אחר` (Other) for the 14 missing responses as the selected preparation rule. This is a modeling assignment, not evidence that the participant reported another payer. Retain all populated codes, including the nine responses that contradict the employment-status routing rule, and preserve those inconsistencies in the audit table. One-hot encode the resulting nominal categories without giving their numbers a quantitative meaning. [scikit-learn OneHotEncoder](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html).

In [29]:
# Can salary source add occupational context without treating category IDs as quantities? Encode it categorically and retain the limits of the missing-value rule.
salary_categories = salary_source.fillna(4).astype("Int64").astype("string")

salary_encoder = OneHotEncoder(categories=[["1", "2", "3", "4"]], handle_unknown="ignore", sparse_output=True)
salary_encoder.fit(salary_categories.loc[feature_fit_rows].to_frame("MakorSachar"))

salary_matrix = salary_encoder.transform(salary_categories.to_frame("MakorSachar"))
salary_features = pd.DataFrame.sparse.from_spmatrix(salary_matrix, index=survey.index, columns=salary_encoder.get_feature_names_out(["MakorSachar"]))
prepared_data = prepared_data.drop(columns="MakorSachar").join(salary_features)
field_encoders["MakorSachar"] = salary_encoder
print("MakorSachar one-hot matrix:", salary_matrix.shape)

MakorSachar one-hot matrix: (200, 4)


### Analysis of TeudaGvoha column

The metadata records the highest certificate or degree. Codes 1–7 cover school qualifications through a doctorate, while 8 means another certificate and 9 means no certificate. The entire code list therefore cannot be treated as one increasing numeric measure. Use one-hot encoding to preserve the qualification categories.

Inspect their frequencies and the number of unanswered qualifications before constructing the categorical input.

In [30]:
# Which qualifications were reported and which responses are absent? Inspect the field before deciding how to represent missingness.
qualifications = survey["TeudaGvoha"]
display(qualifications.value_counts(dropna=False).sort_index().rename("Records"))

TeudaGvoha
1.0     3
2.0    12
3.0    57
4.0    58
5.0    25
6.0    39
7.0     4
NaN     2
Name: Records, dtype: int64

**Conclusion and preparation:** The field contains 198 recorded qualifications and two blanks. Codes 1–7 identify specific certificates, code 8 covers any other certificate, and code 9 denotes no certificate. For this preparation, assign the two missing qualification responses to code 9, interpreting the omission as no certificate reported. This is an explicit interpretation of nonresponse. All populated answers remain unchanged. One-hot encode the categories so their numerical labels do not impose an ordering. [scikit-learn OneHotEncoder](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html).

Use the codebook categories explicitly so an allowed response retains its own column even when absent from a training fold.

In [31]:
# Can qualifications help distinguish occupations without assuming an unanswered field means no qualification? Encode categories while identifying the substitution assumption.
qualification_categories = qualifications.fillna(9).astype("Int64").astype("string")

qualification_encoder = OneHotEncoder(categories=[["1", "2", "3", "4", "5", "6", "7", "8", "9"]], handle_unknown="ignore", sparse_output=True)
qualification_encoder.fit(qualification_categories.loc[feature_fit_rows].to_frame("TeudaGvoha"))

qualification_matrix = qualification_encoder.transform(qualification_categories.to_frame("TeudaGvoha"))
qualification_features = pd.DataFrame.sparse.from_spmatrix(qualification_matrix, index=survey.index, columns=qualification_encoder.get_feature_names_out(["TeudaGvoha"]))
prepared_data = prepared_data.drop(columns="TeudaGvoha").join(qualification_features)
field_encoders["TeudaGvoha"] = qualification_encoder
print("TeudaGvoha one-hot matrix:", qualification_matrix.shape)

TeudaGvoha one-hot matrix: (200, 9)


### Analysis of shnotlimud column

The metadata defines years of schooling as an integer measurement. The preparation specification sets an admissible range of 8–80 years and converts values outside that range to missing. Apply this explicit rule to the recorded measurement, then learn any replacement value from training records.

Inspect the observed distribution to identify the entries affected by the specified range.

In [32]:
# Are schooling years plausible, missing, or unusually low? Inspect the observed distribution before applying numerical bounds.
school_years = survey["shnotlimud"]
display(school_years.value_counts(dropna=False).sort_index().rename("Records"))

shnotlimud
7.0      2
8.0      1
10.0     6
11.0     5
12.0    53
13.0    25
14.0    24
15.0    30
16.0    11
17.0    10
18.0    16
20.0     7
22.0     8
NaN      2
Name: Records, dtype: int64

**Conclusion and preparation:** There are 198 recorded values between 7 and 22 years and two blank answers. The two seven-year values fall below the specified 8–80 preparation range and become missing, giving four missing measurements. Zero would assert no schooling and would violate that range. Use the median of admissible complete-label training records as a central replacement without using the occupation label to estimate schooling. Fit the replacement within each training fold during validation. Preserve the reported values in the source data. [Median imputation](https://scikit-learn.org/stable/modules/impute.html#univariate-feature-imputation)

In [33]:
# How can models receive a complete schooling feature without learning an imputation value from evaluation records? Fit the median on the designated fitting rows.
schooling_values = school_years.where(school_years.between(8, 80)).to_frame("shnotlimud")

from sklearn.impute import SimpleImputer
schooling_imputer = SimpleImputer(strategy="median")
schooling_imputer.fit(schooling_values.loc[feature_fit_rows])

prepared_data["shnotlimud"] = schooling_imputer.transform(schooling_values).ravel()
field_encoders["shnotlimud"] = schooling_imputer
print("Schooling values replaced:", int(schooling_values["shnotlimud"].isna().sum()))
print("Training median:", schooling_imputer.statistics_[0])

Schooling values replaced: 4
Training median: 14.0


### Analysis of Gil column

The metadata defines age as an integer measurement. The preparation specification admits ages from 18 to 120 and converts values outside that range to missing. Age remains numeric because differences between recorded ages have a quantitative meaning.

Inspect its distribution to determine whether the rule removes any observed measurements and how many replacements will be needed.

In [34]:
# Does age provide useful context, and which values are missing or outside the selected range? Inspect before numerical preparation.
age = survey["Gil"]
display(age.value_counts(dropna=False).sort_index().rename("Records"))

Gil
22.0     7
23.0     4
24.0     5
25.0    16
27.0     4
28.0     6
29.0     3
30.0     2
31.0    15
32.0    15
33.0     4
35.0    14
37.0     6
38.0     5
39.0     1
40.0     8
42.0    10
43.0     1
44.0    10
45.0     6
46.0     8
48.0     8
52.0     8
55.0    15
58.0     5
59.0     1
60.0     2
65.0     4
78.0     5
NaN      2
Name: Records, dtype: int64

**Conclusion and preparation:** All 198 recorded ages, ranging from 22 to 78, satisfy the specified 18–120 range. Two answers are blank. Zero is not a plausible age under this specification, and dropping those records would discard their remaining information. Replace the two blanks with the median age fitted on complete-label training records. This supplies a central numeric input without claiming to recover the participant's actual age. Refit the replacement within each training fold during validation.

In [35]:
# How can missing ages be represented consistently without fitting on evaluation data? Use the designated fitting population for imputation.
age_values = age.where(age.between(18, 120)).to_frame("Gil")

age_imputer = SimpleImputer(strategy="median")
age_imputer.fit(age_values.loc[feature_fit_rows])

prepared_data["Gil"] = age_imputer.transform(age_values).ravel()
field_encoders["Gil"] = age_imputer
print("Ages replaced:", int(age_values["Gil"].isna().sum()))
print("Training median:", age_imputer.statistics_[0])

Ages replaced: 2
Training median: 37.0


### Preparation of TarichHathala column

The metadata identifies the date the participant started the current role. The useful measurement is years on the job. Under the assignment's immediate-processing interpretation, tenure runs from that starting date to the processing date. Because no survey or last-update date is supplied, use the current execution date as that reference.

Record the reference date so the calculation is reproducible and state this substitution in the presentation. The model receives elapsed years rather than the calendar date.

In [36]:
# What does the role-start date tell us about experience, and what reference date is available? Compute elapsed time while distinguishing it from survey-time tenure.
start_dates = pd.to_datetime(survey["TarichHathala"])

reference_date = pd.Timestamp.now(tz="UTC").normalize().tz_localize(None)
years_on_job = (reference_date - start_dates).dt.days / 365.25

print("Tenure reference date:", reference_date.date())
display(years_on_job.agg(["count", "min", "max", "median"]).rename("Years on the job"))

Tenure reference date: 2026-11-06


count     200.000000
min         0.418891
max        27.419576
median      4.599589
Name: Years on the job, dtype: float64

**Conclusion and preparation:** All 200 records yield a tenure value. Convert elapsed days to years using 365.25 days per year and retain the resulting continuous measurement. In the prepared table, `TarichHathala` contains years on the job, while the original table retains the starting date. The execution date substitutes for the unavailable survey date and must accompany the reported results.

In [37]:
# How can duration values remain comparable within this run? Record the single reference date used for every conversion.
prepared_data["TarichHathala"] = years_on_job

tenure_reference_date = reference_date.date().isoformat()

### Analysis of SemelAnafSofi column

The metadata defines a four-character industry classification. Its digits identify categories in a hierarchy rather than a numeric magnitude, and `X` denotes incomplete coding. Leading zeros therefore belong to the code representation.

Inspect the short stored values against Excel's number format before restoring a zero. Preserve partial codes. The preparation specification permits this field as a predictor only when the code is available at prediction time, which the supplied assignment does not establish.

In [38]:
# Are industry codes categorical identifiers and are any incomplete? Inspect their format and coverage before encoding.
industry_codes = survey["SemelAnafSofi"].astype("string").str.strip().str.replace(r"\.0$", "", regex=True)

from openpyxl import load_workbook
source_workbook = load_workbook(io.BytesIO(raw), read_only=True, data_only=True)
source_sheet = source_workbook["Survey Dataset"]
industry_position = list(next(source_sheet.values)).index("SemelAnafSofi") + 1
short_industry_codes = industry_codes.str.fullmatch(r"[0-9]{1,3}", na=False)
industry_storage = survey.loc[short_industry_codes, ["ID", "SemelAnafSofi"]].copy()
industry_storage["Excel format"] = [source_sheet.cell(int(index) + 2, industry_position).number_format for index in industry_storage.index]
display(industry_storage)
source_workbook.close()

display(industry_codes.loc[industry_codes.str.contains("X", na=False)].value_counts().rename("Records"))

,ID,SemelAnafSofi,Excel format
29,30.000000,123,0000
61,62.000000,123,0000
92,93.000000,123,0000
124,125.000000,123,0000
150,151.000000,124,0000


SemelAnafSofi
XXXX    6
851X    1
852X    1
3XXX    1
Name: Records, dtype: Int64

**Conclusion and preparation:** Five numeric values use Excel's `0000` format and require a leading zero to reproduce their four-position codes. Nine records contain `X`: six `XXXX` and one each of `851X`, `852X`, and `3XXX`. These are partial classifications, not empty cells, so preserve every known and unresolved position. Restore leading zeros and one-hot encode the resulting categorical values. Keep the industry matrix separate from the primary predictors because its availability before occupation prediction is not established.

In [39]:
# Does industry classification add independent context, and will it be available for new cases? Prepare it separately pending that decision.
industry_values = industry_codes.where(~industry_codes.str.fullmatch(r"[0-9]{1,4}", na=False), industry_codes.str.zfill(4)).to_frame("SemelAnafSofi")

industry_encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=True)
industry_encoder.fit(industry_values.loc[feature_fit_rows])
industry_matrix = industry_encoder.transform(industry_values)
industry_features = pd.DataFrame.sparse.from_spmatrix(industry_matrix, index=survey.index, columns=industry_encoder.get_feature_names_out(["SemelAnafSofi"]))

prepared_data = prepared_data.drop(columns="SemelAnafSofi")
field_encoders["SemelAnafSofi"] = industry_encoder
print("SemelAnafSofi one-hot matrix:", industry_matrix.shape)

SemelAnafSofi one-hot matrix: (200, 42)


### Analysis of SemelMishlachSofi column

The assignment identifies this four-character occupation code as the target. Codes containing `X` were not fully classified by the professional coders. They cannot supply complete labels for supervised training or full-code accuracy, but their known digits must be retained alongside later predictions.

Count complete labels, partial patterns, and records per complete class to establish the training population and its class support.

In [40]:
# How many occupation codes are fully known, and are classes frequent enough to evaluate? Count complete labels and rare classes first.
occupation_codes = survey["SemelMishlachSofi"].astype("string").str.strip().str.replace(r"\.0$", "", regex=True)
occupation_codes = occupation_codes.where(~occupation_codes.str.fullmatch(r"[0-9]{1,4}", na=False), occupation_codes.str.zfill(4))

complete_target = occupation_codes.str.fullmatch(r"[0-9]{4}", na=False)
class_counts = occupation_codes.loc[complete_target].value_counts()
display(class_counts.rename("Records"))
display(occupation_codes.loc[~complete_target].value_counts().rename("Incomplete records"))
display(pd.Series({"Complete labels": int(complete_target.sum()), "Incomplete labels": int((~complete_target).sum()), "Complete classes": class_counts.size, "Singleton classes": int(class_counts.eq(1).sum())}))

SemelMishlachSofi
5223    14
9112    12
2512    10
5120     7
2511     7
5414     7
5412     7
5230     6
5322     6
2631     6
5142     6
9211     5
8183     5
2342     5
5131     5
2635     5
3512     4
3122     4
2514     4
7414     4
4313     4
2655     4
2120     3
3211     3
7127     3
2131     3
8212     2
7412     2
3323     2
3513     2
9412     2
2142     2
9329     2
2262     2
4211     2
3311     1
2113     1
4227     1
3354     1
5153     1
3332     1
2431     1
7126     1
1212     1
1330     1
1345     1
1221     1
2411     1
2423     1
7522     1
8342     1
2330     1
5222     1
2412     1
3334     1
7413     1
3312     1
5419     1
4413     1
Name: Records, dtype: Int64

SemelMishlachSofi
1XXX    4
XXXX    4
335X    1
Name: Incomplete records, dtype: Int64

Complete labels      191
Incomplete labels      9
Complete classes      59
Singleton classes     24
dtype: int64

**Conclusion and preparation:** The 191 complete labels cover 59 classes, including 24 represented by one record. Nine records have partial labels: four `1XXX`, four `XXXX`, and one `335X`. Use complete codes as categorical targets and reserve the nine partial records for prediction. Preserve known prefixes for comparison with the eventual predicted codes. Singleton classes constrain validation coverage and must remain visible in evaluation reporting.

In [41]:
# Which occupation digits are known versus unresolved? Use complete codes for supervised training and preserve partial codes for consistency checks.
prepared_data["SemelMishlachSofi"] = occupation_codes
training_ids = survey.loc[complete_target, "ID"].astype("int64")
prediction_ids = survey.loc[~complete_target, "ID"].astype("int64")
y_train = occupation_codes.loc[complete_target].copy()
partial_targets = occupation_codes.loc[~complete_target].copy()

known_prefixes = partial_targets.str.split("X").str[0]
display(pd.DataFrame({"ID": prediction_ids, "Recorded target": partial_targets, "Known prefix": known_prefixes}))

,ID,Recorded target,Known prefix
17,18,1XXX,1
44,45,XXXX,
49,50,1XXX,1
80,81,1XXX,1
107,108,XXXX,
112,113,1XXX,1
141,142,XXXX,
164,165,335X,335
192,193,XXXX,


## Do feature values stay within one code digit or cross between digits?

For `SemelAnafSofi` and `SemelMishlachSofi`, examine each of the four digit positions separately. At a given position, a recurring feature value is **exclusive** when all of its observed occurrences have the same digit. It **crosses digits** when it occurs with more than one digit. Each heatmap cell shows the percentage of distinct recurring values observed in that digit group that are exclusive to it. Darker color means greater separation. Values observed only once are reported separately because one observation cannot demonstrate a repeatable relationship. A grey cell has no recurring values to assess.

The analysis compares the 16 other survey fields, including the other code. It uses recorded values after the dictionary substitutions and reference expansion, before learned encodings. Apply the field-specific missing-value rules for management, employment status, salary source, and qualification before comparison. Other missing responses supply no value for this comparison. ID is excluded because it identifies records rather than describing a participant’s work. Numeric values are compared exactly, so these plots test repeated-value separation, not a linear association.

A fixed group split reserves the final test before this investigation. The plots use development records so any resulting feature decision can be evaluated without consulting held-out labels. Incomplete occupation labels remain prediction records. Within development, an unknown `X` digit is excluded only from that position’s analysis. Leading zeros remain part of the code.

A positional digit is not a complete category definition. Occupation prefixes of one through four digits form the ISCO hierarchy. The Israeli industry classification uses a lettered section and numeric levels of two, three, and four digits, so its first numeric digit must not be called a formal section. The assignment does not specify a classification edition, and this investigation does not assign unverified code titles. [ILO occupation hierarchy](https://isco.ilo.org/en/isco-08/) and [Israel Central Bureau of Statistics industry structure](https://www.cbs.gov.il/he/mediarelease/DocLib/2012/300/29_12_300b.pdf).

In [ ]:
# Repeated text profiles can inflate random-split scores; keep identical profiles together when evaluating new wording.
from sklearn.model_selection import GroupShuffleSplit

CV_FOLDS = 3
TEST_SIZE = 0.22

text_fields = ["ShemAvoda", "SugAvoda", "ShemMachlaka", "SugMachlaka", "EzoAvoda", "TeurPeula", "TeurTafkid"]
labse_fields = ["ShemAvoda", "SugMachlaka", "EzoAvoda", "TeurPeula"]
reference_fields = {"SugMachlaka": "ShemMachlaka", "TeurPeula": "EzoAvoda", "TeurTafkid": "TeurPeula"}
raw_records = data.loc[data["ID"].notna()].copy()
raw_predictors = raw_records.drop(columns=["ID", "SemelAnafSofi", "SemelMishlachSofi"])
profile_text = raw_records[text_fields].astype("string").apply(lambda column: column.str.strip().fillna(""))
profile_keys = profile_text.apply(tuple, axis=1)
profile_groups = pd.Series(pd.factorize(profile_keys)[0], index=raw_records.index)

labelled_predictors = raw_predictors.loc[complete_target]
labelled_targets = occupation_codes.loc[complete_target]
labelled_groups = profile_groups.loc[complete_target]
development_positions, test_positions = next(GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=SEED).split(labelled_predictors, labelled_targets, labelled_groups))
development_predictors = labelled_predictors.iloc[development_positions]
development_targets = labelled_targets.iloc[development_positions]
development_groups = labelled_groups.iloc[development_positions]
test_predictors = labelled_predictors.iloc[test_positions]
test_targets = labelled_targets.iloc[test_positions]
validation_splits = list(GroupKFold(n_splits=CV_FOLDS, shuffle=True, random_state=SEED).split(development_predictors, development_targets, development_groups))

results_directory = Path("classification_results")
results_directory.mkdir(exist_ok=True)
split_assignments = raw_records[["ID"]].copy()
split_assignments["Group"] = profile_groups
split_assignments["Population"] = "Incomplete label"
split_assignments.loc[development_predictors.index, "Population"] = "Development"
split_assignments.loc[test_predictors.index, "Population"] = "Held-out test"
split_assignments.to_csv(results_directory / "split_assignments.csv", index=False, encoding="utf-8-sig")
display(pd.DataFrame({"Records": [len(development_targets), len(test_targets), int((~complete_target).sum())], "Complete classes": [development_targets.nunique(), test_targets.nunique(), 0]}, index=["Development", "Held-out test", "Incomplete labels"]))

In [ ]:
# Can identical descriptions have different occupation labels? Inspect conflicts before attributing every disagreement to model failure.
analysis_values = raw_records.loc[development_predictors.index].drop(columns="ID").copy()
for field in ["YeshuvAvoda"] + text_fields:
    analysis_values[field] = analysis_values[field].astype("string").str.strip().map(transform_text, na_action="ignore").replace("", pd.NA)
for field, previous_field in reference_fields.items():
    analysis_values.loc[analysis_values[field].eq('כנ"ל').fillna(False), field] = analysis_values[previous_field]
analysis_values["SemelAnafSofi"] = industry_values.loc[analysis_values.index, "SemelAnafSofi"]
analysis_values["SemelMishlachSofi"] = occupation_codes.loc[analysis_values.index]
analysis_values["TarichHathala"] = years_on_job.loc[analysis_values.index]
analysis_values["MenahelEtMi"] = analysis_values["MenahelEtMi"].fillna(0).astype("int64")
analysis_values["MaamadAvoda"] = analysis_values["MaamadAvoda"].fillna(0).astype("int64")
analysis_values["MakorSachar"] = analysis_values["MakorSachar"].fillna(4).astype("int64")
analysis_values["TeudaGvoha"] = analysis_values["TeudaGvoha"].fillna(9).astype("int64")

consistency_rows = []
value_relationships = []
for code_field in ["SemelAnafSofi", "SemelMishlachSofi"]:
    for position in range(1, 5):
        digit = analysis_values[code_field].str[position - 1]
        known_digit = digit.str.fullmatch(r"[0-9]", na=False)
        digit_values = digit.loc[known_digit].astype(int)
        digit_range = range(int(digit_values.min()), int(digit_values.max()) + 1)
        for field in analysis_values.columns.drop(code_field):
            pairs = pd.DataFrame({"Digit": digit_values, "Value": analysis_values.loc[known_digit, field].astype("string")}).dropna()
            counts = pd.crosstab(pairs["Value"], pairs["Digit"]).reindex(columns=digit_range, fill_value=0)
            totals = counts.sum(axis=1)
            distinct_digits = counts.gt(0).sum(axis=1)
            recurring = totals.ge(2)
            for value in counts.index:
                observed = counts.loc[value]
                value_relationships.append({"Code field": code_field, "Position": position, "Feature": field, "Value": value, "Occurrences": int(totals.loc[value]), "Digits": ",".join(map(str, observed.index[observed.gt(0)])), "Digit count": int(distinct_digits.loc[value]), "Recurring": bool(recurring.loc[value]), "Exclusive": bool(recurring.loc[value] and distinct_digits.loc[value] == 1)})
            for number in digit_range:
                present = counts[number].gt(0)
                repeated_values = int((present & recurring).sum())
                exclusive_values = int((present & recurring & distinct_digits.eq(1)).sum())
                percentage = 100 * exclusive_values / repeated_values if repeated_values else np.nan
                consistency_rows.append({"Code field": code_field, "Position": position, "Digit": number, "Feature": field, "Group records": int(digit_values.eq(number).sum()), "Observed records": int(counts[number].sum()), "Distinct values": int(present.sum()), "Singleton values": int((present & ~recurring).sum()), "Recurring values": repeated_values, "Exclusive recurring values": exclusive_values, "Exclusive percent": percentage})
consistency = pd.DataFrame(consistency_rows)
value_relationships = pd.DataFrame(value_relationships)
output = Path("data_preparation_results")
output.mkdir(exist_ok=True)
consistency.to_csv(output / "digit_consistency.csv", index=False, encoding="utf-8-sig")
value_relationships.to_csv(output / "digit_value_relationships.csv", index=False, encoding="utf-8-sig")

from matplotlib.colors import LinearSegmentedColormap
from IPython.display import Image
plot_directory = Path("plots")
plot_directory.mkdir(exist_ok=True)
consistency_colors = LinearSegmentedColormap.from_list("survey_consistency", ["#F8F7F1", "#A8BCA8", "#234737"])
consistency_colors.set_bad("#DCE0D4")
for code_field in ["SemelAnafSofi", "SemelMishlachSofi"]:
    figure, axes = plt.subplots(4, 1, figsize=(18, 22), layout="constrained")
    for position, axis in enumerate(axes, start=1):
        rows = consistency.loc[consistency["Code field"].eq(code_field) & consistency["Position"].eq(position)]
        matrix = rows.pivot(index="Digit", columns="Feature", values="Exclusive percent").reindex(columns=analysis_values.columns.drop(code_field))
        group_sizes = rows.groupby("Digit")["Group records"].first()
        image = axis.imshow(matrix.to_numpy(), cmap=consistency_colors, vmin=0, vmax=100, aspect="auto")
        axis.set_xticks(range(len(matrix.columns)), matrix.columns, rotation=35, ha="right", fontsize=9)
        axis.set_yticks(range(len(matrix)), [f"{digit}  (n={group_sizes.loc[digit]})" for digit in matrix.index])
        axis.set_title(f"{code_field} | digit position {position}", loc="left", color="#253D31", fontweight="bold")
        axis.set_ylabel("Digit and group size")
        for row in range(len(matrix)):
            for column in range(len(matrix.columns)):
                value = matrix.iat[row, column]
                label = f"{value:.0f}%" if pd.notna(value) else "—"
                axis.text(column, row, label, ha="center", va="center", fontsize=8, color="white" if pd.notna(value) and value >= 60 else "#253D31")
    figure.colorbar(image, ax=axes, label="Recurring values confined to this digit (%)", shrink=0.65)
    figure.suptitle("Repeated feature values: separation versus crossing between digits", fontsize=16, color="#253D31")
    figure.savefig(plot_directory / f"{code_field}_digit_consistency.png", dpi=140, facecolor="#F8F7F1")
    display(Image(filename=str(plot_directory / f"{code_field}_digit_consistency.png")))
    plt.close(figure)

supported_consistency = consistency.loc[consistency["Recurring values"].ge(3)].sort_values(["Exclusive percent", "Recurring values"], ascending=[False, False])
display(supported_consistency.groupby(["Code field", "Position"], sort=False).head(5))
crossing_examples = value_relationships.loc[value_relationships["Recurring"] & value_relationships["Digit count"].gt(1)].sort_values(["Digit count", "Occurrences"], ascending=[False, False])
display(crossing_examples.groupby(["Code field", "Position"], sort=False).head(5))

**Conclusion and preparation:** Work-description values show substantially more separation between occupation digits than the coded employment or salary categories. For example, among development records with first occupation digit `2`, all 14 recurring `EzoAvoda` values are exclusive to that digit. The same holds for all nine recurring work descriptions under first digit `5`. These are observed repeated-value relationships, not proof that a value determines an occupation in new surveys. Several 100% cells have few supported values, which is why the denominators and singleton counts accompany the plots.

The fields still provide different information. Employment and salary categories cross occupation digits, so they cannot replace the text descriptions, but crossing alone does not establish that they are useless in combination with other predictors. Keep their field-specific encodings. No new predictor is created from occupation digits because those digits belong to the target. Industry digits remain descriptive analysis until industry-code availability at prediction is established. The evidence supports retaining the separate text fields and reporting occupation accuracy at progressively longer code prefixes.

## Prepared inputs

Each predictor has now been transformed within its own section. The primary input contains TF-IDF term weights, nominal one-hot encodings, the specified management level, completed schooling and age measurements, and years on the job. ID remains a record key and the occupation code remains the target. Industry is encoded separately because its availability before occupation prediction is not established. No missingness indicator features are added.

The following matrices use complete-label records to fit numerical encoders. Later classifier validation must start from source records and refit every learned transformation within the training fold, including workplace matching candidates, vocabulary, IDF, categorical mappings, and numerical replacements.

In [ ]:
# Are all feature blocks and target labels aligned to the same records? Assemble them with consistent row indices before training.
from scipy import sparse
predictor_fields = prepared_data.columns.drop(["ID", "SemelMishlachSofi"]).tolist()
X_train = prepared_data.loc[complete_target, predictor_fields].copy()
X_prediction = prepared_data.loc[~complete_target, predictor_fields].copy()

training_matrix = sparse.csr_matrix(X_train.astype(pd.SparseDtype("float64", 0)).sparse.to_coo())
prediction_matrix = sparse.csr_matrix(X_prediction.astype(pd.SparseDtype("float64", 0)).sparse.to_coo())
feature_names = np.asarray(predictor_fields)
industry_training_matrix = industry_matrix[complete_target.to_numpy()]
industry_prediction_matrix = industry_matrix[(~complete_target).to_numpy()]

display(pd.DataFrame({"Records": [len(X_train), len(X_prediction)], "Numerical features": [training_matrix.shape[1], prediction_matrix.shape[1]]}, index=["Complete target", "Incomplete target"]))

### Save the completed preparation

Save the numerical inputs, fitted field encoders, original workplace transformation output, and salary-routing discrepancies. The reference date accompanies the tenure calculation. These artifacts preserve completed preparation while classifier and representation selection remain a separate evaluation step.

In [ ]:
# Can the analysis be reproduced and its assumptions inspected? Save the prepared data and feature decisions.
import joblib
import sklearn
prepared_data.to_csv(output / "prepared_survey.csv", index=False, encoding="utf-8-sig")
pd.concat([training_ids.rename("ID"), X_train, y_train.rename("SemelMishlachSofi")], axis=1).to_csv(output / "training_inputs.csv", index=False, encoding="utf-8-sig")
pd.concat([prediction_ids.rename("ID"), X_prediction, partial_targets.rename("Recorded_target"), known_prefixes.rename("Known_prefix")], axis=1).to_csv(output / "prediction_inputs.csv", index=False, encoding="utf-8-sig")
sparse.save_npz(output / "training_features.npz", training_matrix)
sparse.save_npz(output / "prediction_features.npz", prediction_matrix)
sparse.save_npz(output / "industry_training_features.npz", industry_training_matrix)
sparse.save_npz(output / "industry_prediction_features.npz", industry_prediction_matrix)

pd.Series(feature_names, name="Feature").to_csv(output / "feature_names.csv", index=False, encoding="utf-8-sig")
pd.Series(industry_encoder.get_feature_names_out(["SemelAnafSofi"]), name="Feature").to_csv(output / "industry_feature_names.csv", index=False, encoding="utf-8-sig")
salary_eligibility_issues.to_csv(output / "preparation_audit.csv", index=False, encoding="utf-8-sig")
review.to_csv(output / "workplace_transformations.csv", index=False, encoding="utf-8-sig")
joblib.dump(field_encoders, output / "feature_encoders.joblib")

manifest = {"records": len(survey), "training_records": len(X_train), "prediction_records": len(X_prediction), "features": len(feature_names), "training_ids": training_ids.tolist(), "prediction_ids": prediction_ids.tolist(), "tfidf_normalization": shem_avoda_vectorizer.norm, "tfidf_smooth_idf": False, "tenure_reference_date": tenure_reference_date, "tenure_definition": "Elapsed days from role start to the current processing date divided by 365.25.", "tenure_assumption": "Immediate processing. The current date substitutes for the unavailable survey or last-update date.", "management_missing_value": 0, "employment_missing_category": 0, "salary_missing_category": 4, "qualification_missing_category": 9, "qualification_missing_assumption": "The participant left the self-completed question blank because there was no certificate to report.", "schooling_range": [8, 80], "age_range": [18, 120], "added_missingness_indicators": False, "industry_in_primary_model": False, "classifier_trained": False, "sklearn_version": sklearn.__version__, "pandas_version": pd.__version__}
(output / "preparation_manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
print("Prepared numerical inputs and field encoders saved.")

## Classifier evaluation

The preparation above documents each field separately. Evaluation repeats those same decisions using only the training records of each fold. The displayed matrices are not reused for validation because their vocabularies and medians were fitted on all complete labels.

The assignment allows resubmission but provides no separate participant identifier or update timestamp. Identical profiles across the seven descriptive fields are therefore kept together when splitting. This is a conservative similarity group, not a claim that those records belong to one person. A fixed group split reserves 22% of groups for one final test. Ten group folds on the remaining records select both the classifier and representation. All 9 incomplete labels are excluded from fitting and scoring. Some complete classes have only one example, so stratification cannot make every class learnable in every fold. The outputs report unseen test classes explicitly.

The main metric is macro F1, giving each represented occupation equal weight. Accuracy reports record-level agreement. Neither measures correctness against independently adjudicated truth because the assignment states that manual coding is imperfect. [GroupKFold](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GroupKFold.html) and [preprocessing without leakage](https://scikit-learn.org/stable/common_pitfalls.html#data-leakage).

In [ ]:
# Compare simple and complex classifiers because sparse occupational descriptions may not reward model complexity.
additional_packages = {"xgboost": "xgboost==3.0.5", "lightgbm": "lightgbm==4.6.0", "catboost": "catboost==1.2.8"}
for package, requirement in additional_packages.items():
    if importlib.util.find_spec(package) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", requirement])

import time
import tempfile
from sklearn.base import BaseEstimator, TransformerMixin, ClassifierMixin
from sklearn.model_selection import GroupShuffleSplit, GridSearchCV, ParameterGrid
from sklearn.preprocessing import MinMaxScaler, normalize
from sklearn.cluster import KMeans
from sklearn.metrics import make_scorer, calinski_harabasz_score, confusion_matrix
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier, VotingClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

cache_directory = tempfile.TemporaryDirectory()
preparation_cache = joblib.Memory(cache_directory.name, verbose=0)

### Fit field transformations within each training fold

The dictionary preserves non-disclosure as text and maps equivalent wording to `חסוי`. References still expand from the preceding answer in the same row. LaBSE replaces full Latin-containing values using the closest Hebrew value above the similarity threshold. Training records supply its Hebrew candidates. For validation or prediction, only those training candidates and the incoming Latin-containing values are passed to the function. Incoming Hebrew values cannot enlarge the candidate dictionary.

Each field keeps its own vocabulary. Nominal fields retain separate one-hot encoders. Management remains ordered, while schooling and age use their stated ranges and training medians. Numeric features are scaled using training minima and maxima because their units otherwise dominate distances and regularization. Tenure uses the recorded processing date. Industry code is excluded from the primary classifier because its availability before occupation coding has not been established.

In [ ]:
# Fit text vocabularies and structured-data preprocessing within each training fold to avoid validation leakage.
def match_training_candidates(values, candidates):
    latin_values = values.loc[values.str.contains(r"[A-Za-z]", na=False)]
    transformed = values.copy()
    if len(latin_values) and len(candidates):
        combined = pd.concat([pd.Series(candidates, dtype="string"), latin_values.reset_index(drop=True)], ignore_index=True)
        matched = transform_column_with_labse(combined)
        transformed.loc[latin_values.index] = matched.iloc[len(candidates):].to_numpy()
    return transformed

@preparation_cache.cache
def prepare_fold_text(training_rows, incoming_rows):
    candidate_names = {}
    prepared_frames = []
    for position, records in enumerate([training_rows, incoming_rows]):
        prepared = pd.DataFrame(index=records.index)
        for field in text_fields:
            values = records[field].astype("string").str.strip().map(transform_text, na_action="ignore").fillna("")
            if field in reference_fields:
                values.loc[values.eq('כנ"ל')] = prepared[reference_fields[field]]
            if field in labse_fields:
                if position == 0:
                    candidate_names[field] = sorted(values.loc[values.str.contains(r"[א-ת]", na=False) & ~values.str.contains(r"[A-Za-z]", na=False)].unique())
                values = match_training_candidates(values, candidate_names[field])
            prepared[field] = values
        prepared_frames.append(prepared)
    return prepared_frames

def prepare_structured_fields(records):
    nominal = pd.DataFrame(index=records.index)
    nominal["YeshuvAvoda"] = records["YeshuvAvoda"].astype("string").str.strip().map(transform_text, na_action="ignore")
    nominal["MaamadAvoda"] = records["MaamadAvoda"].fillna(0).astype("Int64").astype("string")
    nominal["MakorSachar"] = records["MakorSachar"].fillna(4).astype("Int64").astype("string")
    nominal["TeudaGvoha"] = records["TeudaGvoha"].fillna(9).astype("Int64").astype("string")
    numeric = pd.DataFrame(index=records.index)
    numeric["MenahelEtMi"] = records["MenahelEtMi"].fillna(0)
    numeric["shnotlimud"] = records["shnotlimud"].where(records["shnotlimud"].between(8, 80))
    numeric["Gil"] = records["Gil"].where(records["Gil"].between(18, 120))
    numeric["TarichHathala"] = (reference_date - pd.to_datetime(records["TarichHathala"])).dt.days / 365.25
    return nominal, numeric

@preparation_cache.cache
def fit_field_clusters(matrix):
    unit_vectors = normalize(matrix, norm="l2").toarray()
    populated = np.any(unit_vectors != 0, axis=1)
    training_vectors = unit_vectors[populated]
    distinct_count = np.unique(training_vectors, axis=0).shape[0]
    candidates = []
    fitted = []
    for count in range(2, min(100, distinct_count - 1) + 1):
        clusterer = KMeans(n_clusters=count, n_init=10, random_state=SEED)
        labels = clusterer.fit_predict(training_vectors)
        candidates.append({"Clusters": count, "CH": calinski_harabasz_score(training_vectors, labels)})
        fitted.append(clusterer)
    choice = int(np.argmax([entry["CH"] for entry in candidates]))
    return fitted[choice], candidates

class SurveyFeatures(TransformerMixin, BaseEstimator):
    def __init__(self, normalization=None, add_clusters=False):
        self.normalization = normalization
        self.add_clusters = add_clusters

    def fit(self, records, target=None):
        self.training_rows_ = records.copy()
        prepared_text, _ = prepare_fold_text(records, records)
        self.vectorizers_ = {}
        self.clusterers_ = {}
        self.cluster_search_ = []
        self.feature_names_ = []
        for field in text_fields:
            vectorizer = clone(shem_avoda_vectorizer).set_params(norm=self.normalization)
            matrix = vectorizer.fit_transform(prepared_text[field])
            self.vectorizers_[field] = vectorizer
            self.feature_names_.extend(field + "__" + vectorizer.get_feature_names_out())
            if self.add_clusters:
                clusterer, search_rows = fit_field_clusters(matrix)
                self.clusterers_[field] = clusterer
                self.cluster_search_.extend([{"Field": field, **row} for row in search_rows])
                self.feature_names_.extend([field + "__cluster_" + str(label) for label in range(clusterer.n_clusters)])
        nominal, numeric = prepare_structured_fields(records)
        self.nominal_encoders_ = {}
        category_definitions = {"MaamadAvoda": [["0", "1", "2", "3", "4", "5", "6", "7"]], "MakorSachar": [["1", "2", "3", "4"]], "TeudaGvoha": [["1", "2", "3", "4", "5", "6", "7", "8", "9"]]}
        for field in nominal:
            encoder = OneHotEncoder(categories=category_definitions.get(field, "auto"), handle_unknown="ignore", sparse_output=False)
            encoder.fit(nominal[[field]])
            self.nominal_encoders_[field] = encoder
            self.feature_names_.extend(encoder.get_feature_names_out([field]))
        self.numeric_imputer_ = SimpleImputer(strategy="median")
        imputed = self.numeric_imputer_.fit_transform(numeric)
        self.numeric_scaler_ = MinMaxScaler(clip=True).fit(imputed)
        self.feature_names_.extend(numeric.columns)
        return self

    def transform(self, records):
        _, prepared_text = prepare_fold_text(self.training_rows_, records)
        blocks = []
        for field in text_fields:
            matrix = self.vectorizers_[field].transform(prepared_text[field])
            blocks.append(matrix.toarray())
            if self.add_clusters:
                normalized = normalize(matrix, norm="l2").toarray()
                memberships = np.zeros((len(records), self.clusterers_[field].n_clusters))
                populated = np.any(normalized != 0, axis=1)
                memberships[np.flatnonzero(populated), self.clusterers_[field].predict(normalized)[populated]] = 1
                blocks.append(memberships)
        nominal, numeric = prepare_structured_fields(records)
        blocks.extend([encoder.transform(nominal[[field]]) for field, encoder in self.nominal_encoders_.items()])
        blocks.append(self.numeric_scaler_.transform(self.numeric_imputer_.transform(numeric)))
        return np.concatenate(blocks, axis=1)

    def get_feature_names_out(self, input_features=None):
        return np.asarray(self.feature_names_)

### Why compare these representations and models?

TF-IDF turns each questionnaire text field into lexical evidence. L2 normalization makes vector lengths comparable; it does not remove sparsity. Different Hebrew and English terms may still describe the same concept, so any LaBSE-based normalization must be checked for incorrect substitutions.

We keep L2-normalized TF-IDF fixed and ask whether adding learned text-cluster membership helps. To keep the experiment interpretable and affordable, we compare three complementary classifiers—logistic regression, linear SVC, and random forest—with two parameter values each. A majority-class baseline provides context. Three description-group folds select the model by macro-F1; the held-out test is used only after selection. This search has 13 candidates and 39 CV fits.

SVD and supervised feature selection are separate representation experiments, not automatically applied here. They must be fitted within each training fold and shown to improve grouped validation before adoption.


In [ ]:
# Compare a linear baseline, a margin classifier, and a tree model; each tests a different way to separate occupations.
model_families = {
    "Logistic regression": (LogisticRegression(max_iter=1000, class_weight="balanced", random_state=SEED), {"classifier__C": [0.1, 1]}),
    "Linear SVC": (LinearSVC(class_weight="balanced", random_state=SEED), {"classifier__C": [0.1, 1]}),
    "Random forest": (RandomForestClassifier(n_estimators=80, class_weight="balanced", random_state=SEED, n_jobs=1), {"classifier__min_samples_leaf": [1, 2]})
}

# Test whether text-derived clusters add useful evidence beyond the same L2-normalized TF-IDF features.
parameter_grids = []
candidate_descriptions = []
for add_clusters in [False, True]:
    for family, (classifier, parameters) in model_families.items():
        parameter_grids.append({"features": [SurveyFeatures(normalization="l2", add_clusters=add_clusters)], "classifier": [classifier], **parameters})
        candidate_descriptions.extend([{"Classifier": family, "Normalization": "l2", "Clusters added": add_clusters, "Parameters": json.dumps(combination)} for combination in ParameterGrid(parameters)])

# Keep an untuned majority baseline to establish whether learned features improve on a trivial prediction.
parameter_grids.append({"features": [SurveyFeatures(normalization="l2", add_clusters=False)], "classifier": [DummyClassifier(strategy="most_frequent")]})
candidate_descriptions.append({"Classifier": "Majority baseline", "Normalization": "l2", "Clusters added": False, "Parameters": "{}"})
display(pd.DataFrame([{"Classifier": name, "Parameters": str(grid)} for name, (_, grid) in model_families.items()]))

# Select the best configuration on three description-group folds; keep the held-out test outside model selection.
assert len(validation_splits) == CV_FOLDS, "Validation splits must match CV_FOLDS."
model_pipeline = Pipeline([("features", SurveyFeatures()), ("classifier", LogisticRegression())], memory=preparation_cache)
scoring = {"macro_f1": make_scorer(f1_score, average="macro", zero_division=0), "accuracy": "accuracy"}
search = GridSearchCV(model_pipeline, parameter_grids, cv=validation_splits, scoring=scoring, refit="macro_f1", n_jobs=3, return_train_score=False, error_score="raise", verbose=1)
search_start = time.perf_counter()
with warnings.catch_warnings(record=True) as recorded_warnings:
    warnings.simplefilter("always")
    with joblib.parallel_backend("threading"):
        search.fit(development_predictors, development_targets)
search_seconds = time.perf_counter() - search_start
joblib.dump(search, results_directory / "search_checkpoint.joblib")
print("Search seconds:", round(search_seconds, 1))

# Preserve each candidate's fold scores so the interviewer can see why the winner was selected.
model_results = pd.concat([pd.DataFrame(candidate_descriptions), pd.DataFrame(search.cv_results_).drop(columns=["params"] + [name for name in search.cv_results_ if name.startswith("param_")])], axis=1)
model_results.to_csv(results_directory / "cross_validation_results.csv", index=False, encoding="utf-8-sig")
warning_counts = pd.Series([str(item.message) for item in recorded_warnings], dtype="string").value_counts().rename("Count")
warning_counts.to_csv(results_directory / "training_warnings.csv", encoding="utf-8-sig")
best_configuration = model_results.loc[search.best_index_]
family_results = model_results.sort_values("mean_test_macro_f1", ascending=False, kind="stable").drop_duplicates(["Classifier", "Normalization", "Clusters added"])
display(family_results[["Classifier", "Normalization", "Clusters added", "mean_test_macro_f1", "std_test_macro_f1", "mean_test_accuracy"]])
display(best_configuration[["Classifier", "Normalization", "Clusters added", "Parameters", "mean_test_macro_f1", "mean_test_accuracy"]])


### Read the three-fold comparison

Each row shows development macro-F1 for one classifier and representation setting. Compare the same folds, inspect fold-to-fold variation, and select the highest mean macro-F1 without looking at held-out labels. The cluster comparison is exploratory: it measures whether adding those features helps under this validation protocol, not whether clusters have a causal effect.


In [ ]:
# Do aggregate scores hide failures on rare occupations? Examine fold variation and class-level errors.
family_results["Representation"] = family_results["Normalization"] + family_results["Clusters added"].map({False: " | TF-IDF", True: " | TF-IDF + clusters"})
comparison = family_results.pivot(index="Classifier", columns="Representation", values="mean_test_macro_f1")
figure, axis = plt.subplots(figsize=(12, 7), layout="constrained")
image = axis.imshow(comparison, cmap=consistency_colors, vmin=0, vmax=max(0.5, comparison.max().max()), aspect="auto")
axis.set_xticks(range(len(comparison.columns)), comparison.columns, rotation=20, ha="right")
axis.set_yticks(range(len(comparison.index)), comparison.index)
for row in range(len(comparison)):
    for column in range(len(comparison.columns)):
        value = comparison.iat[row, column]
        axis.text(column, row, f"{value:.3f}" if pd.notna(value) else "—", ha="center", va="center", color="white" if pd.notna(value) and value > 0.3 else "#253D31")
axis.set_title("Three-fold validation macro F1", loc="left", fontweight="bold")
figure.colorbar(image, ax=axis, label="Macro F1")
figure.savefig(plot_directory / "classifier_comparison.png", dpi=160, facecolor="#F8F7F1")
display(Image(filename=str(plot_directory / "classifier_comparison.png")))
plt.close(figure)

fold_overview = pd.DataFrame([{"Fold": fold, "Training records": len(fit_rows), "Validation records": len(validation_rows), "Unseen-class records": int((~development_targets.iloc[validation_rows].isin(development_targets.iloc[fit_rows])).sum()), "Macro F1": search.cv_results_[f"split{fold - 1}_test_macro_f1"][search.best_index_], "Accuracy": search.cv_results_[f"split{fold - 1}_test_accuracy"][search.best_index_]} for fold, (fit_rows, validation_rows) in enumerate(validation_splits, start=1)])
fold_overview.to_csv(results_directory / "selected_fold_results.csv", index=False)
display(fold_overview)



### Evaluate the selected model once on the reserved test

The three grouped development folds select the representation, classifier, and hyperparameters. Only that selected pipeline is evaluated on held-out records. This avoids using test performance to choose among candidates. Compare its predictions with the majority baseline, inspect errors and occupation-code prefixes, and keep unseen classes in the evaluation denominator.


In [ ]:
# The test set answers one question: does the development-selected model generalize to unseen description groups?
# Keep this helper for later prediction diagnostics; Linear SVC has no native probability output.
def prediction_scores(fitted_pipeline, inputs):
    if hasattr(fitted_pipeline, "predict_proba"):
        return fitted_pipeline.predict_proba(inputs), "Uncalibrated probability"
    return None, "Not available"


### Selected classifier on the reserved test

The overall configuration was selected using development cross-validation. Report its accuracy, macro F1, and agreement at one through four occupation-code digits. Classes absent from development remain in the denominator. These measurements compare predictions with the supplied manual labels, which are not independently adjudicated ground truth.

In [ ]:
# Does the selected approach outperform a simple baseline on unseen groups? Compare held-out predictions and inspect disagreements.
selected_pipeline = search.best_estimator_
test_predictions = np.asarray(selected_pipeline.predict(test_predictors)).ravel()
baseline = DummyClassifier(strategy="most_frequent").fit(development_predictors, development_targets)
baseline_predictions = baseline.predict(test_predictors)
held_out_predictions = raw_records.loc[test_predictors.index, ["ID", "ShemAvoda", "EzoAvoda"]].copy()
held_out_predictions["Recorded target"] = test_targets
held_out_predictions["Prediction"] = test_predictions
held_out_predictions["Class seen in development"] = test_targets.isin(development_targets)
held_out_predictions["Correct"] = test_predictions == test_targets
held_out_metrics = pd.DataFrame([{"Model": best_configuration["Classifier"], "Accuracy": accuracy_score(test_targets, test_predictions), "Macro F1": f1_score(test_targets, test_predictions, average="macro", zero_division=0)}, {"Model": "Majority baseline", "Accuracy": accuracy_score(test_targets, baseline_predictions), "Macro F1": f1_score(test_targets, baseline_predictions, average="macro", zero_division=0)}])
held_out_metrics.to_csv(results_directory / "held_out_metrics.csv", index=False)
prefix_metrics = pd.DataFrame([{"Digits": width, "Agreement": np.mean([predicted[:width] == actual[:width] for predicted, actual in zip(test_predictions, test_targets)])} for width in [1, 2, 3, 4]])
prefix_metrics.to_csv(results_directory / "hierarchical_agreement.csv", index=False)
classification_details = pd.DataFrame(classification_report(test_targets, test_predictions, output_dict=True, zero_division=0)).T
classification_details.to_csv(results_directory / "classification_report.csv", encoding="utf-8-sig")
display(held_out_metrics)
display(prefix_metrics)
print("Test records with classes absent from development:", int((~held_out_predictions["Class seen in development"]).sum()))


test_scores, test_score_kind = prediction_scores(selected_pipeline, test_predictors)
if test_scores is not None:
    top_three = selected_pipeline.classes_[np.argsort(-test_scores, axis=1, kind="stable")[:, :3]]
    held_out_predictions["Top three"] = [", ".join(values) for values in top_three]
    held_out_predictions["Top-three correct"] = [actual in alternatives for actual, alternatives in zip(test_targets, top_three)]
held_out_predictions["Score kind"] = test_score_kind
held_out_predictions.to_csv(results_directory / "held_out_predictions.csv", index=False, encoding="utf-8-sig")
display(held_out_predictions)
display(classification_details)

display(raw_records.loc[held_out_predictions.index[~held_out_predictions["Correct"]], ["ID", "EzoAvoda", "TeurPeula", "TeurTafkid", "SemelMishlachSofi"]].join(held_out_predictions[["Prediction", "Class seen in development"]]))

### What does separating repeated descriptions change?

Compare the same fixed logistic classifier under random-row and description-group partitions. This diagnostic does not select the model. Description overlap and unseen classes are counted alongside the scores because both change with the split. A score difference therefore cannot be attributed entirely to leakage. Repeated wording does not prove that records belong to the same person.

In [ ]:
# Do sparse text features need reduction? Compare unchanged TF-IDF, selected features, and SVD using grouped development folds only.
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.model_selection import cross_validate
from sklearn.preprocessing import Normalizer

# L2 controls vector length; SVD compresses correlated dimensions; chi-square selection retains original, interpretable features.
representation_folds = list(GroupKFold(n_splits=3).split(development_predictors, development_targets, development_groups))
representation_candidates = {
    "Original TF-IDF": "passthrough",
    "Selected original features": SelectKBest(score_func=chi2, k=64),
    "SVD components": Pipeline([("svd", TruncatedSVD(n_components=32, random_state=SEED)), ("l2", Normalizer())]),
}
representation_results = []
for name, reducer in representation_candidates.items():
    pipeline = Pipeline([("features", SurveyFeatures(normalization="l2", add_clusters=False)), ("representation", reducer), ("classifier", LogisticRegression(max_iter=1500, class_weight="balanced", random_state=SEED))], memory=preparation_cache)
    scores = cross_validate(pipeline, development_predictors, development_targets, cv=representation_folds, scoring={"macro_f1": make_scorer(f1_score, average="macro", zero_division=0), "accuracy": "accuracy"}, n_jobs=1, error_score="raise")
    representation_results.append({"Representation": name, "Macro F1": scores["test_macro_f1"].mean(), "Macro F1 std": scores["test_macro_f1"].std(), "Accuracy": scores["test_accuracy"].mean()})
representation_results = pd.DataFrame(representation_results).sort_values("Macro F1", ascending=False)
display(representation_results)
# This diagnostic does not select the final model; retain a reduction method only if its grouped-validation benefit is reproducible.


In [ ]:
# Compare random-row and grouped splits to assess sensitivity to repeated descriptions, not to quantify leakage directly.
from sklearn.model_selection import KFold
split_diagnostics = []
for split_name, splitter in [("Random rows", KFold(n_splits=CV_FOLDS, shuffle=True, random_state=SEED)), ("Description groups", GroupKFold(n_splits=CV_FOLDS, shuffle=True, random_state=SEED))]:
    partitions = list(splitter.split(labelled_predictors, labelled_targets, labelled_groups))
    fixed_pipeline = Pipeline([("features", SurveyFeatures()), ("classifier", LogisticRegression(max_iter=1500, class_weight="balanced", random_state=SEED))], memory=preparation_cache)
    with joblib.parallel_backend("threading"):
        diagnostic_predictions = cross_val_predict(fixed_pipeline, labelled_predictors, labelled_targets, cv=partitions, n_jobs=4)
    overlap = sum(labelled_groups.iloc[validation].isin(labelled_groups.iloc[training]).sum() for training, validation in partitions)
    unseen = sum((~labelled_targets.iloc[validation].isin(labelled_targets.iloc[training])).sum() for training, validation in partitions)
    split_diagnostics.append({"Split": split_name, "Accuracy": accuracy_score(labelled_targets, diagnostic_predictions), "Macro F1": f1_score(labelled_targets, diagnostic_predictions, average="macro", zero_division=0), "Description overlap records": int(overlap), "Unseen-class records": int(unseen)})
split_diagnostics = pd.DataFrame(split_diagnostics)
split_diagnostics.to_csv(results_directory / "split_diagnostics.csv", index=False)
display(split_diagnostics)

**Conclusion:** For the fixed logistic classifier, random-row validation gave 82.2% accuracy and macro F1 of 0.520. Keeping description groups separate gave 69.6% and 0.398. Under the random split, 89 validation records shared a description profile with training records. Group separation removed that overlap, while records from unseen classes increased from 26 to 44. Both factors changed, so the score difference does not isolate a leakage effect. It shows why the split definition materially affects the reported performance. The selected classifier remains determined by development group cross-validation.


### Final fit and predictions

Refit the single selected configuration on all complete labels and predict the nine incomplete records. Retain each original partial code. A compatible alternative uses only classes learned during training and is reported separately from the unconstrained prediction. Agreement with known digits does not establish correctness of the unknown digits. These predictions support examination by a coder, and the simulated sample does not establish a safe automation threshold.

In [ ]:
# How can incomplete occupation codes be predicted without contradicting known digits? Refit on complete labels and check each partial-code constraint.
final_pipeline = clone(selected_pipeline).set_params(memory=None).fit(labelled_predictors, labelled_targets)
unresolved_predictors = raw_predictors.loc[~complete_target]
final_predictions = np.asarray(final_pipeline.predict(unresolved_predictors)).ravel()
prediction_results = raw_records.loc[~complete_target, ["ID", "ShemAvoda", "EzoAvoda"]].copy()
prediction_results["Recorded target"] = occupation_codes.loc[~complete_target]
prediction_results["Known prefix"] = known_prefixes
prediction_results["Predicted occupation"] = final_predictions
prediction_results["Prefix agrees"] = [prediction.startswith(prefix) for prediction, prefix in zip(final_predictions, known_prefixes)]


partial_scores, partial_score_kind = prediction_scores(final_pipeline, unresolved_predictors)
trained_codes = final_pipeline.classes_
ranking_rows = []
for row_number, (index, record) in enumerate(prediction_results.iterrows()):
    compatible = np.array([all(original == "X" or original == proposed for original, proposed in zip(record["Recorded target"], code)) for code in trained_codes])
    ranking = np.argsort(-partial_scores[row_number], kind="stable") if partial_scores is not None else np.array([], dtype=int)
    compatible_ranking = ranking[compatible[ranking]]
    ranking_rows.append({"ID": int(record["ID"]), "Top three": ", ".join(trained_codes[ranking[:3]]), "Score kind": partial_score_kind, "Maximum score": float(partial_scores[row_number].max()) if partial_scores is not None else np.nan, "Digit-compatible alternative": trained_codes[compatible_ranking[0]] if len(compatible_ranking) else pd.NA, "Compatible learned classes": int(compatible.sum())})
prediction_results = prediction_results.merge(pd.DataFrame(ranking_rows), on="ID", how="left")
prediction_results.to_csv(results_directory / "incomplete_label_predictions.csv", index=False, encoding="utf-8-sig")
display(prediction_results)

### Apply the two voting methods to one new questionnaire

Construct one example questionnaire and train fresh soft and hard ensembles using their respective CV-selected settings. Both predict the same record. The example has no known occupation label and therefore supplies no accuracy estimate. The hard ensemble returns a class vote without an invented probability score.

In [ ]:
# Can a new questionnaire follow the same preparation steps as training data? Demonstrate the end-to-end prediction path.
new_record = raw_predictors.iloc[:0].reindex([0]).copy()
new_record.loc[0, ["YeshuvAvoda", "ShemAvoda", "SugAvoda", "EzoAvoda", "TeurPeula", "TeurTafkid"]] = ["חיפה", "חברת תוכנה חדשה", "פיתוח תוכנה", "מפתחת תוכנה", "כתיבת קוד ובדיקת תוכנה", "מפתחת"]
new_record.loc[0, ["MaamadAvoda", "MakorSachar", "TeudaGvoha", "shnotlimud", "Gil", "TarichHathala"]] = [1, 1, 5, 16, 30, "2024-01-01"]

ensemble_demonstration = []
for name, voting in [("Soft voting", "soft"), ("Hard voting", "hard")]:
    chosen_index = model_results.loc[model_results["Classifier"].eq(name), "mean_test_macro_f1"].idxmax()
    configuration = model_results.loc[chosen_index]
    normalization = None if configuration["Normalization"] == "None" else configuration["Normalization"]
    ensemble = Pipeline([("features", SurveyFeatures(normalization=normalization, add_clusters=bool(configuration["Clusters added"]))), ("classifier", build_ensemble(voting))], memory=preparation_cache)
    ensemble.set_params(**json.loads(configuration["Parameters"])).fit(labelled_predictors, labelled_targets)
    prediction = np.asarray(ensemble.predict(new_record)).ravel()[0]
    scores, kind = prediction_scores(ensemble, new_record)
    ranking = np.argsort(-scores[0], kind="stable")[:3] if scores is not None else np.array([], dtype=int)
    ensemble_demonstration.append({"Ensemble": name, "Prediction": prediction, "Top three": ", ".join(ensemble.classes_[ranking]), "Score kind": kind})
ensemble_demonstration = pd.DataFrame(ensemble_demonstration)
ensemble_demonstration.to_csv(results_directory / "new_record_predictions.csv", index=False, encoding="utf-8-sig")
display(new_record)
display(ensemble_demonstration)

### Reproducible results and implementation

The outputs retain the split assignments, every grid candidate and fold score, held-out predictions, partial-label predictions, and the selected configuration. The stored pipeline requires the class definitions and LaBSE model in this notebook. It is an assignment artifact, not a standalone deployment package.

For operational deployment, collect a participant key, submission timestamp, and coder identity, then create an independently adjudicated evaluation sample. Validate by participant and time, assess class coverage, calibrate scores, and select an automation threshold from measured error and coverage. Keep a human coding route for unsupported occupations and conflicting evidence. The current experiment uses dense matrices for compatibility across libraries at 200 rows. Larger workloads require sparse processing and measured memory, training, and prediction costs.

In [ ]:
# What must be saved so later predictions use the same feature definitions? Persist the fitted model and preprocessing artifacts.
joblib.dump(final_pipeline, results_directory / "selected_pipeline.joblib")
pd.Series(final_pipeline.named_steps["features"].get_feature_names_out(), name="Feature").to_csv(results_directory / "selected_feature_names.csv", index=False, encoding="utf-8-sig")
final_training_features = final_pipeline.named_steps["features"].transform(labelled_predictors)
final_prediction_features = final_pipeline.named_steps["features"].transform(unresolved_predictors)
sparse.save_npz(results_directory / "selected_training_features.npz", sparse.csr_matrix(final_training_features))
sparse.save_npz(results_directory / "selected_prediction_features.npz", sparse.csr_matrix(final_prediction_features))
run_manifest = {"seed": SEED, "training_records": len(labelled_predictors), "development_records": len(development_predictors), "test_records": len(test_predictors), "incomplete_records": len(prediction_results), "folds": len(validation_splits), "candidates": len(model_results), "selected_classifier": best_configuration["Classifier"], "selected_normalization": best_configuration["Normalization"], "clusters_added": bool(best_configuration["Clusters added"]), "selected_parameters": json.loads(best_configuration["Parameters"]), "validation_macro_f1": float(best_configuration["mean_test_macro_f1"]), "test_accuracy": float(held_out_metrics.iloc[0]["Accuracy"]), "test_macro_f1": float(held_out_metrics.iloc[0]["Macro F1"]), "final_features": final_training_features.shape[1], "tenure_reference_date": tenure_reference_date, "labse_alpha": 0.5, "management_missing_value": 0, "employment_missing_category": 0, "salary_missing_category": 4, "qualification_missing_category": 9, "labse_fields": labse_fields, "text_transformations": text_transformations, "industry_in_primary_model": False, "sklearn_version": sklearn.__version__, "search_seconds": search_seconds}
(results_directory / "run_manifest.json").write_text(json.dumps(run_manifest, ensure_ascii=False, indent=2), encoding="utf-8")
print("Selected pipeline, validation results, and final predictions saved.")

## Results and conclusion

The configuration selected by development cross-validation determines the final model. Compare its held-out agreement with the majority baseline and inspect performance at each occupation-code prefix. The representation comparison uses the same folds for every classifier. Final predictions for incomplete labels are distinct from measured test performance because their complete reference labels are unavailable.


In [ ]:
# Which errors occur at the occupation-prefix level and which representations contribute? Inspect these diagnostics to guide improvement.
figure, axes = plt.subplots(1, 2, figsize=(12, 4.5), layout="constrained")
bars = axes[0].bar(prefix_metrics["Digits"].astype(str), prefix_metrics["Agreement"], color="#234737")
axes[0].bar_label(bars, labels=[f"{value:.1%}" for value in prefix_metrics["Agreement"]], padding=4)
axes[0].set(xlabel="Occupation prefix length", ylabel="Agreement", ylim=(0, 1.08), title="Selected model: held-out agreement")
positions = np.arange(len(held_out_metrics))
for offset, metric, color in [(-0.18, "Accuracy", "#234737"), (0.18, "Macro F1", "#946E37")]:
    bars = axes[1].bar(positions + offset, held_out_metrics[metric], width=0.34, color=color, label=metric)
    axes[1].bar_label(bars, fmt="%.3f", padding=4)
axes[1].set(xticks=positions, xticklabels=held_out_metrics["Model"], ylim=(0, 1.08), title="Selected model and majority baseline")
axes[1].legend(frameon=False)
figure.savefig(plot_directory / "held_out_summary.png", dpi=180, facecolor="#F8F7F1")
display(Image(filename=str(plot_directory / "held_out_summary.png")))
plt.close(figure)

# Compare candidate families using development CV only; the test remains a one-time check of the winner.
classifier_comparison = model_results.sort_values("mean_test_macro_f1", ascending=False, kind="stable").drop_duplicates("Classifier")
figure, axis = plt.subplots(figsize=(11, 6), layout="constrained")
positions = np.arange(len(classifier_comparison))
bars = axis.barh(positions, classifier_comparison["mean_test_macro_f1"], color="#234737")
axis.bar_label(bars, fmt="%.3f", padding=4)
axis.set(yticks=positions, yticklabels=classifier_comparison["Classifier"], xlim=(0, 1.05), xlabel="Development CV macro F1", title="Best development configuration per classifier")
axis.invert_yaxis()
figure.savefig(plot_directory / "selected_classifier_comparison.png", dpi=180, facecolor="#F8F7F1")
display(Image(filename=str(plot_directory / "selected_classifier_comparison.png")))
plt.close(figure)


**Interpretation after execution:** The reduced search compares 13 candidates across three grouped folds (39 CV fits). The tables and figures above report the actual winning model and scores from this run; results from the earlier ten-fold, larger search are not valid for this configuration. The held-out evaluation measures agreement with supplied codes, not independently verified occupation accuracy. Inspect errors by occupation prefix and whether a class appeared in development. For partially coded records, distinguish unrestricted predictions from candidates compatible with the recorded digits; missing digits have no verified reference labels. Before operational use, obtain adjudicated labels, representative test data, and explicit human-review criteria.
